In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional, Literal, Dict, Union, Any, Tuple, Set
import re
import logging
import datetime
import time
import random
import json
import jsonlines
import os
import colorlog
from tqdm import tqdm
from opensearchpy import OpenSearch
from rapidfuzz import process, fuzz
from openai import Client
import concurrent.futures
from functools import partial
from threading import Semaphore


def setup_colored_logging():
  
    handler = colorlog.StreamHandler()
    handler.setFormatter(
        colorlog.ColoredFormatter(
            "%(log_color)s%(asctime)s - %(name)s - %(levelname)s - %(message)s",
            log_colors={
                "DEBUG": "cyan",
                "INFO": "green",
                "WARNING": "yellow",
                "ERROR": "red",
                "CRITICAL": "red,bg_white",
            },
        )
    )

    logger = colorlog.getLogger("annual_report_analyzer")
    logger.handlers = []
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)

    return logger


logger = setup_colored_logging()

llm_api_url = ""
client = Client(
    base_url=llm_api_url, api_key=""
)

OPENSEARCH_CONFIG = {
    "hosts": [
        {"host": "", "port": }
    ],
    "http_auth": ("", ""),
    "use_ssl": True,
    "verify_certs": True,
    "ssl_show_warn": False,
    "timeout": 120,
    "retry_on_timeout": True,
    "max_retries": 3,
}

opensearch_client = OpenSearch(**OPENSEARCH_CONFIG)
INDEX_NAME = ""

company_match_cache = {}
company_to_sha1 = {}

api_log_file = "all_api_responses_sequential.jsonl"
step_counter = 0

api_semaphore = Semaphore(10)


def rate_limited_api_call(func, *args, **kwargs):
    with api_semaphore:
        result = func(*args, **kwargs)
        time.sleep(0.1)
        return result


class QuestionClassifier(BaseModel):
    reasoning: str = Field(
        ...,
        description="Detailed reasoning process analyzing whether this question requires comparing multiple companies, including linguistic markers, specific comparison phrases, and context clues that support the classification",
    )

    is_comparison: bool = Field(
        ...,
        description="Boolean indicator of whether the question requires comparing information across multiple companies (True) or focuses on a single company (False)",
    )

    comparison_aspect: Optional[str] = Field(
        None,
        description="The specific metric, attribute, or information dimension being compared between companies (e.g., revenue, employees, market share, profitability)",
    )

    comparison_type: Optional[
        Literal["highest", "lowest", "difference", "similarity", "other"]
    ] = Field(
        None,
        description="The nature of the comparison being requested: 'highest' (which company has the maximum), 'lowest' (which company has the minimum), 'difference' (how companies differ), 'similarity' (how companies are alike), or 'other' for different comparison types",
    )


class CompaniesExtractor(BaseModel):
    reasoning: str = Field(
        ...,
        description="Detailed reasoning about how each company was identified in the question, including exact mentions, inferences, disambiguations, and confidence in each identification",
    )

    companies: List[str] = Field(
        ...,
        description="Complete list of company names that need to be compared, exactly matching names from the provided company list",
    )

    primary_metric: str = Field(
        ...,
        description="The central metric or attribute being compared between companies, precisely as stated or clearly implied in the question",
    )

    metric_type: Optional[
        Literal[
            "total_revenue",
            "total_assets",
            "net_income",
            "employee_count",
            "cash_flow_from_operations",
            "gross_margin_percentage",
            "capital_expenditures",
            "dividend_per_share",
            "executive_compensation",
            "number_of_facilities",
            "number_of_stores",
            "cloud_storage_capacity",
            "customer_base",
            "active_user_count",
            "fleet_size",
            "number_of_patents",
            "number_of_pharmaceutical_patents",
            "number_of_clinics",
            "number_of_healthcare_professionals",
            "non_performing_loan_ratio",
            "co2_emissions",
            "number_of_active_software_licenses",
            "number_of_active_licensing_deals",
            "tech_staff_headcount",
            "renewable_energy_percentage",
            "total_headcount",
            "power_generation_capacity",
            "generic_product_count",
            "rd_spending",
            "number_of_fulfillment_centers",
            "total_deposits",
            "outstanding_insurance_claims",
            "e_commerce_active_customers",
            "number_of_hotels",
            "clinical_trial_sites",
            "number_of_hybrid_models",
            "year_end_user_base",
            "net_interest_margin",
            "market_share",
            "other",
        ]
    ] = Field(
        None,
        description="The standardized category of metric being compared, selected from the predefined list of metrics to enable consistent data retrieval across different companies",
    )


class MetricExtractor(BaseModel):
    reasoning: str = Field(
        ...,
        description="Comprehensive reasoning about the metric identification process, including analysis of the question's phrasing, related terminology, domain context, and justification for selecting this specific metric type over alternatives",
    )

    metric_type: Literal[
        "total_revenue",
        "total_assets",
        "net_income",
        "employee_count",
        "cash_flow_from_operations",
        "gross_margin_percentage",
        "capital_expenditures",
        "dividend_per_share",
        "executive_compensation",
        "number_of_facilities",
        "number_of_stores",
        "cloud_storage_capacity",
        "customer_base",
        "active_user_count",
        "fleet_size",
        "number_of_patents",
        "number_of_pharmaceutical_patents",
        "number_of_clinics",
        "number_of_healthcare_professionals",
        "non_performing_loan_ratio",
        "co2_emissions",
        "number_of_active_software_licenses",
        "number_of_active_licensing_deals",
        "tech_staff_headcount",
        "renewable_energy_percentage",
        "total_headcount",
        "power_generation_capacity",
        "generic_product_count",
        "rd_spending",
        "number_of_fulfillment_centers",
        "total_deposits",
        "outstanding_insurance_claims",
        "e_commerce_active_customers",
        "number_of_hotels",
        "clinical_trial_sites",
        "number_of_hybrid_models",
        "year_end_user_base",
        "net_interest_margin",
        "market_share",
        "unknown",
    ] = Field(
        ...,
        description="The specific standardized metric type that best matches what is being asked for in the question, selected from a comprehensive list of financial, operational, and industry-specific metrics",
    )

    metric_description: str = Field(
        ...,
        description="Clear, concise explanation of what this metric measures, how it's typically calculated, and its significance in understanding company performance",
    )

    domain: Literal["financial", "operations", "corporate", "leadership", "other"] = (
        Field(
            ...,
            description="The broad business domain this metric belongs to, categorizing it into financial performance, operational metrics, corporate actions, leadership information, or other specialized areas",
        )
    )


class DomainClassifier(BaseModel):
    reasoning: str = Field(
        ...,
        description="Detailed analysis explaining why this particular domain was chosen, including specific keywords, context clues, topic correlations, and industry-standard categorization principles that support this domain classification",
    )

    domain: Literal["financial", "operations", "corporate", "leadership", "other"] = (
        Field(
            ...,
            description="The primary business domain that the question falls into: financial (money, performance, metrics), operations (day-to-day activities, facilities, employees), corporate (strategic decisions, company structure), leadership (management, executives), or other",
        )
    )

    domain_description: str = Field(
        ...,
        description="Comprehensive explanation of what this domain encompasses, its boundaries, key components, and typical metrics or information associated with it",
    )

    primary_focus: str = Field(
        ...,
        description="The specific central topic, metric, or information point within the selected domain that is most relevant to answering the question",
    )


class CompanyIdentifier(BaseModel):
    reasoning: str = Field(
        ...,
        description="Detailed reasoning about how the company and domain were identified, including analysis of direct mentions, contextual clues, industry associations, and disambiguation of similar company names",
    )

    company: Optional[str] = Field(
        None,
        description="The primary company that is the focus of the question, exactly matching a name from the provided company list",
    )

    related_companies: Optional[List[str]] = Field(
        None,
        description="Other companies mentioned in the question that are not the primary focus but may be relevant for context",
    )

    domain: Literal["financial", "operations", "corporate", "leadership", "other"] = (
        Field(
            ...,
            description="The primary business domain of the question: financial, operations, corporate, leadership, or other",
        )
    )

    question_user: str = Field(
        ...,
        description="The original question text exactly as asked by the user, preserved for reference",
    )

    rephrased_question: str = Field(
        ...,
        description="A clear, standardized restatement of the question that resolves ambiguities and explicitly identifies the information being sought",
    )

    reasoning_about_query: str = Field(
        ...,
        description="Detailed analysis of the query formation strategy, explaining how the three query expansions were designed to maximize information retrieval success",
    )

    reasoning_step2: str = Field(
        ...,
        description="Strategic explanation of how the expanded queries will identify different relevant passages by targeting diverse vocabulary patterns that might contain the answer",
    )

    query_expansion_1: str = Field(
        ...,
        description="CORE ENTITY EXTRACTION without company names or verbs: Extract the essential subject/entity being asked about in the original question. This should represent the raw 'what' of the search - the specific financial metric, operational fact, or corporate information being sought (e.g., if the question is 'What was Apple's annual revenue in 2022?', the query should be 'annual revenue 2022', not 'what was revenue')",
    )

    query_expansion_2: str = Field(
        ...,
        description="ALTERNATIVE TERMINOLOGY: 2-3 industry-specific synonyms and related terms that express the same concept using different vocabulary that might appear in annual reports (e.g., 'sales turnover earnings income financial performance')",
    )

    query_expansion_3: str = Field(
        ...,
        description="CONTEXTUAL INDICATORS: 4-6 terms that typically surround or indicate the presence of the requested information in annual reports, focusing on report-specific phrasing (e.g., 'fiscal year reported compared previous consolidated')",
    )


class FinancialMetricsData(BaseModel):
    reasoning: str = Field(
        ...,
        description="Detailed explanation of how financial metrics were identified in the text, including explicit quotes, context interpretation, and confidence assessment for each extracted metric",
    )

    reasoning_step2: str = Field(
        ...,
        description="Advanced financial analysis examining the strategic implications of these metrics, including year-over-year trends, industry comparisons, and impacts on company valuation and outlook",
    )

    page_num: int = Field(
        ..., description="Document page number where this information was found"
    )

    total_revenue: Optional[float] = Field(
        None,
        description="Total revenue/sales for the reporting period in millions or billions, representing the gross income from all business activities",
    )

    cash_flow_from_operations: Optional[float] = Field(
        None,
        description="Net cash generated from core business operations, excluding investing and financing activities",
    )

    gross_margin_percentage: Optional[float] = Field(
        None,
        description="Gross profit as a percentage of revenue, indicating efficiency in production and sales processes",
    )

    net_income: Optional[float] = Field(
        None,
        description="Net profit or earnings after all expenses, taxes, and costs have been deducted from revenue",
    )

    total_assets: Optional[float] = Field(
        None,
        description="Combined value of all company-owned resources with economic value",
    )

    capital_expenditures: Optional[float] = Field(
        None,
        description="Funds used to acquire, upgrade, or maintain physical assets like property, buildings, equipment",
    )

    dividend_per_share: Optional[float] = Field(
        None,
        description="Amount of dividends paid to shareholders per outstanding share",
    )

    executive_compensation: Optional[float] = Field(
        None,
        description="Total compensation provided to executive leadership, including salary, bonuses, stock options, and benefits",
    )

    currency: Optional[str] = Field(
        None,
        description="Currency denomination for financial metrics (USD, EUR, GBP, etc.)",
    )

    fiscal_year_end: Optional[str] = Field(
        None,
        description="The date or month when the fiscal year ended for the reported metrics",
    )

    found_data: bool = Field(
        ...,
        description="Boolean indicating whether relevant financial data was found on this page",
    )


class BusinessOperationsData(BaseModel):
    reasoning: str = Field(
        ...,
        description="Detailed explanation of how operational metrics were identified in the text, including explicit quotes, context interpretation, and confidence assessment for each extracted metric",
    )

    reasoning_step2: str = Field(
        ...,
        description="Strategic analysis of operational data examining business implications, efficiency indicators, capacity utilization, operational bottlenecks, and competitive positioning",
    )

    page_num: int = Field(
        ..., description="Document page number where this information was found"
    )

    employee_count: Optional[int] = Field(
        None,
        description="Total number of employees working for the company, including full-time and part-time staff",
    )

    employees_let_go: Optional[int] = Field(
        None,
        description="Number of employees terminated or laid off during the reporting period",
    )

    number_of_facilities: Optional[int] = Field(
        None,
        description="Total count of physical locations operated by the company, including manufacturing plants, offices, and distribution centers",
    )

    number_of_stores: Optional[int] = Field(
        None,
        description="Number of retail locations or storefronts operated by the company",
    )

    cloud_storage_capacity: Optional[float] = Field(
        None, description="Company's cloud storage capacity measured in terabytes (TB)"
    )

    customer_base: Optional[int] = Field(
        None,
        description="Total number of individual or business customers currently served by the company",
    )

    active_user_count: Optional[int] = Field(
        None,
        description="Number of users actively engaging with the company's products or services within the reporting period",
    )

    fleet_size: Optional[int] = Field(
        None,
        description="Total number of vehicles owned or leased by the company for operations",
    )

    number_of_patents: Optional[int] = Field(
        None,
        description="Total count of active patents owned by the company, covering its intellectual property",
    )

    number_of_pharmaceutical_patents: Optional[int] = Field(
        None,
        description="Count of active pharmaceutical-specific patents held by the company",
    )

    number_of_clinics: Optional[int] = Field(
        None,
        description="Number of healthcare clinics owned or operated by the company",
    )

    number_of_healthcare_professionals: Optional[int] = Field(
        None,
        description="Count of doctors, nurses, and other healthcare professionals employed by the company",
    )

    non_performing_loan_ratio: Optional[float] = Field(
        None,
        description="Percentage of loans in default or close to default relative to total loans (for financial institutions)",
    )

    co2_emissions: Optional[float] = Field(
        None,
        description="Carbon dioxide emissions produced by company operations, typically measured in metric tons",
    )

    number_of_active_software_licenses: Optional[int] = Field(
        None,
        description="Count of software licenses currently active or in use by the company or its customers",
    )

    number_of_active_licensing_deals: Optional[int] = Field(
        None,
        description="Number of current licensing agreements where the company is licensing its IP or products to others",
    )

    tech_staff_headcount: Optional[int] = Field(
        None,
        description="Number of employees in technology-related roles, including IT, engineering, and development",
    )

    renewable_energy_percentage: Optional[float] = Field(
        None,
        description="Percentage of the company's energy consumption derived from renewable sources",
    )

    total_headcount: Optional[int] = Field(
        None,
        description="Total number of people employed by the company globally, including all subsidiaries",
    )

    power_generation_capacity: Optional[float] = Field(
        None,
        description="Company's total capacity to generate electricity, measured in megawatts (MW)",
    )

    generic_product_count: Optional[int] = Field(
        None,
        description="Number of generic products manufactured or distributed by the company",
    )

    rd_spending: Optional[float] = Field(
        None,
        description="Amount spent on research and development activities during the reporting period",
    )

    number_of_fulfillment_centers: Optional[int] = Field(
        None,
        description="Count of warehouses or distribution centers used for order processing and fulfillment",
    )

    total_deposits: Optional[float] = Field(
        None,
        description="Total value of customer deposits held by the financial institution",
    )

    outstanding_insurance_claims: Optional[float] = Field(
        None,
        description="Total value of insurance claims filed but not yet settled or paid",
    )

    e_commerce_active_customers: Optional[int] = Field(
        None,
        description="Number of customers who made at least one purchase through the company's e-commerce channels in the reporting period",
    )

    number_of_hotels: Optional[int] = Field(
        None,
        description="Count of hotels owned, managed, or franchised by the company at year-end",
    )

    clinical_trial_sites: Optional[int] = Field(
        None,
        description="Number of locations conducting clinical trials for the company's products",
    )

    number_of_hybrid_models: Optional[int] = Field(
        None,
        description="Count of hybrid vehicle models offered by the automotive company",
    )

    year_end_user_base: Optional[int] = Field(
        None,
        description="Total number of users or customers at the end of the fiscal year",
    )

    net_interest_margin: Optional[float] = Field(
        None,
        description="Difference between interest income and interest expenses as a percentage of average earning assets (for financial institutions)",
    )

    market_share: Optional[float] = Field(
        None,
        description="Company's percentage share of the total market for its products or services",
    )

    found_data: bool = Field(
        ...,
        description="Boolean indicating whether relevant business operations data was found on this page",
    )


class CorporateActionsData(BaseModel):
    reasoning: str = Field(
        ...,
        description="Detailed explanation of how corporate actions were identified in the text, including explicit quotes, context interpretation, and confidence assessment for each extracted action",
    )

    reasoning_step2: str = Field(
        ...,
        description="Strategic analysis examining the business implications of corporate actions, potential market impact, alignment with company strategy, and effects on stakeholders",
    )

    page_num: int = Field(
        ..., description="Document page number where this information was found"
    )

    mergers_acquisitions: Optional[bool] = Field(
        None,
        description="Boolean indicating whether the company mentioned any mergers or acquisitions",
    )

    ma_details: Optional[List[str]] = Field(
        None,
        description="Detailed information about specific mergers and acquisitions, including company names, transaction values, and completion status",
    )

    share_buyback_plan: Optional[bool] = Field(
        None,
        description="Boolean indicating whether the company announced or implemented a share repurchase program",
    )

    dividend_policy_changes: Optional[bool] = Field(
        None,
        description="Boolean indicating whether the company made changes to its dividend distribution policy",
    )

    capital_structure_changes: Optional[bool] = Field(
        None,
        description="Boolean indicating whether the company modified its capital structure, such as debt-to-equity ratio or share classes",
    )

    restructuring_plans: Optional[bool] = Field(
        None,
        description="Boolean indicating whether the company announced plans to reorganize its business units, workforce, or operations",
    )

    new_product_launches: Optional[bool] = Field(
        None,
        description="Boolean indicating whether the company launched new products during the reporting period",
    )

    new_product_names: Optional[List[str]] = Field(
        None, description="Names of specific new products launched by the company"
    )

    last_product_launched: Optional[str] = Field(
        None, description="Name of the most recent product launched by the company"
    )

    esg_initiatives: Optional[bool] = Field(
        None,
        description="Boolean indicating whether the company announced environmental, social, or governance initiatives",
    )

    esg_initiative_details: Optional[List[str]] = Field(
        None,
        description="Specific details about the company's environmental, social, and governance initiatives",
    )

    ongoing_litigation: Optional[bool] = Field(
        None,
        description="Boolean indicating whether the company disclosed any active legal proceedings or disputes",
    )

    litigation_details: Optional[List[str]] = Field(
        None,
        description="Specific details about ongoing legal cases, including parties involved, claims, and potential financial impact",
    )

    found_data: bool = Field(
        ...,
        description="Boolean indicating whether relevant corporate actions data was found on this page",
    )


class LeadershipManagementData(BaseModel):
    reasoning: str = Field(
        ...,
        description="Detailed explanation of how leadership information was identified in the text, including explicit quotes, context interpretation, and confidence assessment for each extracted data point",
    )

    reasoning_step2: str = Field(
        ...,
        description="Analysis of leadership changes and their potential impact on company strategy, operations, culture, and market perception",
    )

    page_num: int = Field(
        ..., description="Document page number where this information was found"
    )

    leadership_positions_changed: Optional[List[str]] = Field(
        None,
        description="List of specific leadership positions that experienced personnel changes during the reporting period",
    )

    executive_appointments: Optional[List[str]] = Field(
        None,
        description="Details of new executive appointments, including names, positions, start dates, and relevant background information",
    )

    executive_departures: Optional[List[str]] = Field(
        None,
        description="Details of executive departures, including names, positions, departure dates, and reasons if provided",
    )

    board_changes: Optional[List[str]] = Field(
        None,
        description="Changes to the board of directors, including new appointments, departures, and changes in board structure",
    )

    organizational_restructuring: Optional[bool] = Field(
        None,
        description="Boolean indicating whether the company underwent significant reorganization of its leadership structure",
    )

    found_data: bool = Field(
        ...,
        description="Boolean indicating whether relevant leadership and management data was found on this page",
    )


class Source(BaseModel):
    """Information source for the answer"""

    document_name: str = Field(
        ..., description="Name of the source document containing the information"
    )

    page_number: int = Field(
        ..., description="Page number where the information was found"
    )

    text_excerpt: str = Field(
        ..., description="Relevant excerpt from the document that supports the answer"
    )

    relevance_score: Optional[float] = Field(
        None,
        description="Numerical score indicating how relevant this source is to the question, from 0 to 1",
    )


class Reference(BaseModel):
    pdf_sha1: str = Field(
        ..., description="SHA1 hash of the PDF file for unique identification"
    )

    page_index: int = Field(..., description="Page index within the document (1-based)")

    excerpt_text: Optional[str] = Field(
        None,
        description="Brief text excerpt from the referenced page that supports the answer",
    )


class CompanyComparisonData(BaseModel):
    reasoning: str = Field(
        ...,
        description="Detailed analysis of the comparison methodology, data normalization approach, and confidence assessment for each company's metrics",
    )

    metric_type: Literal[
        "total_revenue", "total_assets", "net_income", "employee_count", "other"
    ] = Field(
        ..., description="The specific type of metric being compared across companies"
    )

    currency: Optional[str] = Field(
        None,
        description="Currency denomination for financial metrics, ensuring proper comparison",
    )

    companies_data: Dict[str, Optional[float]] = Field(
        ...,
        description="Dictionary mapping company names to their corresponding metric values",
    )

    lowest_company: Optional[str] = Field(
        None, description="Company with the lowest value for the metric being compared"
    )

    highest_company: Optional[str] = Field(
        None, description="Company with the highest value for the metric being compared"
    )

    relative_differences: Optional[Dict[str, float]] = Field(
        None,
        description="Dictionary showing each company's percentage difference from the mean",
    )

    found_data: bool = Field(
        ..., description="Boolean indicating whether usable comparison data was found"
    )


class Answer(BaseModel):
    reasoning_question_analysis: str = Field(
        ...,
        description="Detailed analysis breaking down the question into its component parts, identifying specific entities mentioned, determining exactly what information is being requested, clarifying ambiguities, and identifying the expected answer type and format",
    )

    reasoning_context_evaluation: str = Field(
        ...,
        description="Thorough assessment of whether the provided context contains the necessary information to answer the question, including analysis of relevance, sufficiency, and quality of available data, identifying key information across sources, noting agreements and discrepancies, and evaluating what critical information might be missing",
    )

    reasoning_for_math_calc: Optional[str] = Field(
        None,
        description="Step-by-step mathematical analysis when the question requires numerical computation, showing each calculation explicitly, explaining the reasoning behind each step, the significance of intermediate results, and verifying the final calculation",
    )

    reasoning_source_comparison: Optional[str] = Field(
        None,
        description="Methodical comparison of information from different pages and sources, analyzing consistency, contradictions, differences in specificity, and determining which sources are most authoritative or relevant",
    )

    reasoning_temporal_analysis: Optional[str] = Field(
        None,
        description="Analysis of time-related aspects of the data, including which fiscal periods are covered, how time-dependent terms like 'at year end' are interpreted, and relevant year-over-year or quarter-over-quarter changes",
    )

    reasoning_data_normalization: Optional[str] = Field(
        None,
        description="Explanation of any data normalization performed to ensure fair comparison, such as currency conversion, adjusting for company size, or accounting for different reporting standards",
    )

    reasoning_confidence_assessment: Optional[str] = Field(
        None,
        description="Detailed evaluation of confidence level in the answer, examining specificity of source information, consistency across sources, completeness of available data, and potential alternative interpretations",
    )

    reasoning_final_decision: str = Field(
        ...,
        description="Conclusive evaluation that synthesizes all previous analysis to explain how the final answer was determined, considering alternative interpretations, assessing confidence level based on evidence quality, addressing remaining uncertainties, and justifying why this specific answer is most supported by the available information",
    )

    value: Union[float, str, bool, List[str], Literal["N/A"]] = Field(
        ...,
        description="Answer value according to the question type (number, boolean, name, or names)",
    )

    confidence: float = Field(
        ...,
        ge=0,
        le=1,
        description="Confidence in the answer from 0 to 1, based on comprehensive analysis of evidence quality and completeness",
    )

    sources: Optional[List[Source]] = Field(
        None,
        description="Information sources for the answer, with detailed context from each relevant source",
    )


class OutputAnswer(BaseModel):
    question_text: str = Field(
        ..., description="The original question text exactly as provided"
    )

    value: Union[str, float, bool, List[str]] = Field(
        ...,
        description="The answer value in the appropriate format for the question type",
    )

    confidence: Optional[float] = Field(
        None, description="Confidence score for the answer, from 0 to 1"
    )

    references: List[Reference] = Field(
        default_factory=list,
        description="References to specific document pages that support the answer",
    )


def log_api_call_to_jsonl(
    system_prompt,
    user_prompt,
    response,
    response_type,
    model_name,
    page_num=None,
    company_name=None,
):
    global step_counter
    step_counter += 1

    try:
        response_dict = (
            pydantic_to_dict(response)
            if hasattr(response, "model_dump") or hasattr(response, "dict")
            else str(response)
        )

        log_entry = {
            "step": step_counter,
            "timestamp": datetime.datetime.now().isoformat(),
            "response_type": response_type,
            "model": model_name,
            "company_name": company_name,
            "page_num": page_num,
            "system_prompt": system_prompt[:1000]
            + ("..." if len(system_prompt) > 1000 else ""),
            "user_prompt": user_prompt[:1000]
            + ("..." if len(user_prompt) > 1000 else ""),
            "response": response_dict,
        }

        with jsonlines.open(api_log_file, mode="a") as writer:
            writer.write(log_entry)

        logger.debug(f"Logged API call (step {step_counter}) to {api_log_file}")

    except Exception as e:
        logger.error(f"Error logging API call to JSONL: {str(e)}")


def process_pages_in_batch(extractor_function, pages_data, batch_size=5, max_workers=5):
    results = []

    for i in range(0, len(pages_data), batch_size):
        batch = pages_data[i : i + batch_size]

        logger.info(
            f"Processing batch of {len(batch)} pages (batch {i // batch_size + 1}/{(len(pages_data) + batch_size - 1) // batch_size})"
        )

        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = []
            for page_data in batch:
                future = executor.submit(
                    extractor_function,
                    page_data["text"],
                    page_data["page_number"],
                    page_data["company_name"],
                )
                futures.append(future)

            for future in concurrent.futures.as_completed(futures):
                try:
                    result = future.result()
                    if result.found_data:
                        results.append(result)
                except Exception as e:
                    logger.error(f"Error in batch processing: {str(e)}")

    return results


def find_closest_company_match(
    potential_company: str, company_list: List[str]
) -> Optional[str]:
    global company_match_cache

    if not potential_company or not company_list:
        return None

    if potential_company in company_match_cache:
        return company_match_cache[potential_company]

    if potential_company in company_list:
        company_match_cache[potential_company] = potential_company
        return potential_company

    if potential_company.startswith("Odyssey"):
        odyssey_companies = [c for c in company_list if c.startswith("Odyssey")]

        if "Gold" in potential_company or "Limited" in potential_company:
            for company in odyssey_companies:
                if "Gold" in company or "Limited" in company:
                    company_match_cache[potential_company] = company
                    return company
        elif (
            "Group" in potential_company
            or "Holdings" in potential_company
            or "Inc" in potential_company
        ):
            for company in odyssey_companies:
                if "Group" in company or "Holdings" in company or "Inc" in company:
                    company_match_cache[potential_company] = company
                    return company

    if len(potential_company) >= 4:
        prefix = potential_company[:4].lower()
        matches = [
            company for company in company_list if company.lower().startswith(prefix)
        ]
        if matches:
            best_match = matches[0]
            company_match_cache[potential_company] = best_match
            logger.debug(f"Prefix match for '{potential_company}' -> '{best_match}'")
            return best_match

    try:
        best_match, score = process.extractOne(
            potential_company, company_list, scorer=fuzz.token_sort_ratio
        )

        if score >= 70:
            company_match_cache[potential_company] = best_match
            logger.debug(
                f"Fuzzy company match: '{potential_company}' -> '{best_match}' (score: {score})"
            )
            return best_match
    except Exception as e:
        logger.warning(f"Error in fuzzy matching: {str(e)}")

    company_match_cache[potential_company] = None
    logger.warning(f"No company match found for '{potential_company}'")
    return None


def load_company_metadata(meta_file: str) -> Dict[str, str]:
    logger.info(f"Loading company metadata from {meta_file}")

    try:
        with open(meta_file, "r", encoding="utf-8") as f:
            metadata = json.load(f)

        company_to_sha1 = {}
        for item in metadata:
            company_name = item.get("company_name")
            sha1 = item.get("sha1")
            if company_name and sha1:
                company_to_sha1[company_name] = sha1

        logger.info(f"Created mapping for {len(company_to_sha1)} companies")

        sample_entries = list(company_to_sha1.items())[:3]
        logger.debug(f"Sample company mappings: {sample_entries}")

        return company_to_sha1

    except Exception as e:
        logger.error(f"Error loading company metadata: {str(e)}")
        return {}


def pydantic_to_dict(obj):
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    elif hasattr(obj, "dict"):
        return obj.dict()
    elif isinstance(obj, list):
        return [pydantic_to_dict(item) for item in obj]
    elif isinstance(obj, dict):
        return {key: pydantic_to_dict(value) for key, value in obj.items()}
    else:
        return obj


def get_company_names_from_opensearch() -> List[str]:
    logger.info(f"Retrieving list of all companies from OpenSearch index: {INDEX_NAME}")

    try:
        query = {
            "size": 0,
            "aggs": {
                "unique_companies": {"terms": {"field": "company", "size": 10000}}
            },
        }

        response = opensearch_client.search(index=INDEX_NAME, body=query)

        company_buckets = (
            response.get("aggregations", {})
            .get("unique_companies", {})
            .get("buckets", [])
        )
        company_names = [bucket["key"] for bucket in company_buckets]

        logger.info(f"Found {len(company_names)} unique companies in the index")

        if len(company_names) <= 10:
            logger.info(f"Companies: {', '.join(company_names)}")
        else:
            logger.info(f"First 10 companies: {', '.join(company_names[:10])}...")

        return company_names

    except Exception as e:
        logger.error(f"Error retrieving companies from OpenSearch: {str(e)}")
        return []


def log_api_call(
    system_prompt,
    user_prompt,
    model_name,
    response,
    response_type="general",
    page_num=None,
    company_name=None,
):
    # лог
    log_message = f"""
===== API CALL DETAILS =====
STEP: {step_counter}
TYPE: {response_type}
MODEL: {model_name}
COMPANY: {company_name if company_name else "N/A"}
PAGE: {page_num if page_num is not None else "N/A"}
--- SYSTEM PROMPT (EXCERPT) ---
{system_prompt[:300]}... (truncated)
--- USER PROMPT (EXCERPT) ---
{user_prompt[:300]}... (truncated)
--- RESPONSE (EXCERPT) ---
{str(response)[:200]}... (truncated)
==========================
"""
    logger.debug(log_message)

    log_api_call_to_jsonl(
        system_prompt,
        user_prompt,
        response,
        response_type,
        model_name,
        page_num,
        company_name,
    )


def get_parsed_content(completion, model_class):
    try:
        if hasattr(completion.choices[0].message, "parsed"):
            result = completion.choices[0].message.parsed
            logger.debug(
                f"Received structured response via .parsed: {str(result)[:100]}..."
            )
            return result

        elif hasattr(completion.choices[0].message, "content"):
            content = completion.choices[0].message.content
            logger.debug(f"Received response via .content: {content[:100]}...")

            if content.strip().startswith("{") and content.strip().endswith("}"):
                try:
                    json_data = json.loads(content)
                    parsed_model = model_class(**json_data)
                    logger.debug(
                        f"JSON successfully parsed into model: {str(parsed_model)[:100]}..."
                    )
                    return parsed_model
                except Exception as json_error:
                    logger.error(f"Error parsing JSON response: {str(json_error)}")
                    logger.debug(f"Problematic JSON content: {content}")

            return content

        logger.warning(
            f"Could not extract structured data, returning entire message object"
        )
        return completion.choices[0].message

    except Exception as e:
        logger.error(f"Error extracting data from API response: {str(e)}")
        raise e


def is_comparison_question(question_text: str) -> bool:
    logger.info(f"Classifying question type: {question_text}")

    system_prompt = """
    You are an expert analyst specializing in financial and business document analysis.
    Your task is to determine if a question requires comparing multiple companies or focuses on a single company.
    
    # Comparison Question Characteristics
    Comparison questions typically:
    1. Ask which company among several has the highest/lowest value for a specific metric
       - Example: "Which company had the highest revenue?"
       - Example: "Among companies X, Y, and Z, which had the lowest employee count?"
    
    2. Request explicit comparison between specific companies
       - Example: "How does Company A's profit compare to Company B's?"
       - Example: "What's the difference in market share between Companies X and Y?"
    
    3. Ask for rankings or relative positions of multiple companies
       - Example: "Rank these companies by net income."
       - Example: "List the companies in order of their R&D spending."
    
    4. Inquire about differences or similarities between companies
       - Example: "What are the main differences in ESG initiatives between these companies?"
       - Example: "How similar are these companies' dividend policies?"
    
    5. Use comparative linguistic markers
       - Example: "more than," "less than," "versus," "compared to," "relative to"
       - Example: "Which of the companies," "which among," "between these companies" 
    
    # Single Company Question Characteristics
    Single company questions typically:
    1. Ask about a specific metric or fact for one clearly identified company
       - Example: "What was Apple's annual revenue in 2022?"
       - Example: "Did Tesla announce any new product launches?"
    
    2. Request information about events, policies or actions of a single company
       - Example: "Did Microsoft implement any restructuring plans?"
       - Example: "What ESG initiatives did Amazon undertake?"
    
    3. Ask yes/no questions about a specific company's situation
       - Example: "Does Netflix have any ongoing litigation?"
       - Example: "Did Bank of America change its dividend policy?"
    
    # Your Analysis Task
    1. Carefully analyze the question structure, entities mentioned, and linguistic markers
    2. Determine if the question requires comparing information across multiple companies
    3. Identify the specific metric or aspect being compared (if applicable)
    4. Determine the type of comparison being requested (highest, lowest, difference, etc.)
    5. Provide detailed reasoning for your classification
    """

    user_prompt = f"""
    Analyze the following question and determine if it requires comparing information across multiple companies:
    
    Question: {question_text}
    
    Provide detailed reasoning about why this question does or does not require comparison between companies.
    If it is a comparison question, identify what specific aspect is being compared and what type of comparison is requested.
    """

    max_retries = 3
    retry_count = 0
    base_delay = 5

    while retry_count <= max_retries:
        try:
            completion = rate_limited_api_call(
                client.beta.chat.completions.parse,
                model="large",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.1,
                response_format=QuestionClassifier,
            )

            log_api_call(
                system_prompt,
                user_prompt,
                "large",
                completion,
                response_type="question_classifier",
            )

            result = get_parsed_content(completion, QuestionClassifier)

            if result.is_comparison:
                logger.info(f"Question classified as COMPARISON question")
                logger.info(
                    f"Comparison aspect: {result.comparison_aspect}, type: {result.comparison_type}"
                )
            else:
                logger.info(f"Question classified as SINGLE COMPANY question")

            return result.is_comparison

        except Exception as e:
            error_message = str(e)
            retry_count += 1

            if "Connection error" in error_message and retry_count <= max_retries:
                delay = base_delay * (2 ** (retry_count - 1))

                logger.warning(
                    f"Connection error in question classification. Retrying attempt {retry_count}/{max_retries} after {delay} seconds. Error: {error_message}"
                )
                time.sleep(delay)
            else:
                logger.error(f"Error classifying question with LLM: {error_message}")
                logger.info("Falling back to keyword-based classification")

                comparison_keywords = [
                    "which of",
                    "which company",
                    "which companies",
                    "compare",
                    "comparison",
                    "comparative",
                    "compared to",
                    "highest",
                    "lowest",
                    "greater",
                    "lesser",
                    "more than",
                    "less than",
                    "versus",
                    "vs",
                    "difference between",
                    "similarities between",
                    "rank",
                    "ranking",
                    "relative to",
                    "in order of",
                    "among these companies",
                    "between these companies",
                ]

                for keyword in comparison_keywords:
                    if keyword.lower() in question_text.lower():
                        logger.info(
                            f"Question classified as COMPARISON based on keyword '{keyword}' (fallback)"
                        )
                        return True

                logger.info("Question classified as SINGLE COMPANY (fallback)")
                return False


def extract_company_names_from_question(
    question_text: str, all_company_names: List[str]
) -> List[str]:
    logger.info(f"Extracting companies from comparison question: {question_text}")

    max_companies_in_prompt = 200
    companies_subset = (
        all_company_names[:max_companies_in_prompt]
        if len(all_company_names) > max_companies_in_prompt
        else all_company_names
    )
    companies_formatted = "\n".join(companies_subset)

    if len(all_company_names) > max_companies_in_prompt:
        logger.info(
            f"Using subset of {max_companies_in_prompt} companies in prompt (out of {len(all_company_names)} total)"
        )

    system_prompt = """
    You are an expert analyst specializing in financial and business document analysis.
    Your task is to identify which companies need to be compared based on the question and the provided list of companies.
    
    # Company Identification Guidelines
    
    1. Direct Mentions:
       - Identify companies that are explicitly named in the question
       - Example: "Compare Apple and Google's revenues" → Apple, Google
    
    2. Quoted Names:
       - Pay special attention to companies in quotation marks
       - Example: "Which of these companies had higher assets: 'Alpha Inc.', 'Beta Corp.'?" → Alpha Inc., Beta Corp.
    
    3. Partial Matches:
       - If question contains partial company names, find the full name in the provided list
       - Example: "Compare Microsoft and Facebook" → Microsoft Corporation, Facebook Inc.
    
    4. Lists of Companies:
       - Identify all companies in comma-separated or bulleted lists
       - Example: "Compare the revenues of X, Y, and Z" → All three companies
    
    5. Contextual References:
       - Look for phrases like "the following companies:" followed by a list
       - Example: "Which of the following had the highest revenue: A, B, C?" → A, B, C
    
    # Metric Identification Guidelines
    
    1. Primary Comparison Metric:
       - Identify the exact metric being compared between companies
       - Common metrics: revenue, assets, employees, profit, market share, etc.
    
    2. Metric Type Categorization:
       - Classify the metric into a standardized category from the list provided
       - Example: "annual earnings" → metric_type: "total_revenue"
    
    # Requirements
    
    1. EXACT COMPANY NAME MATCHING:
       - Your final company list MUST use EXACT names from the provided company list
       - Check for exact spelling, punctuation, and abbreviations
    
    2. COMPREHENSIVE EXTRACTION:
       - Identify ALL companies that need to be compared
       - If the question asks "which of X, Y, Z...", all listed companies must be included
    
    3. DETAILED REASONING:
       - Explain your identification process for each company
       - Note any ambiguities or corrections made to match company names
    
    4. PRECISION:
       - Do not add companies that aren't clearly part of the comparison
       - If in doubt about a company name, check against the provided list
    """

    user_prompt = f"""
    Identify the companies being compared in the following question:
    
    Question: {question_text}
    
    Available companies (subset of complete list):
    {companies_formatted}
    
    Task:
    1. List all companies that need to be compared in this question
    2. Identify the primary metric or attribute being compared
    3. Classify the metric into a standardized type if possible
    4. Provide detailed reasoning for your identification process
    
    Ensure your company names exactly match those in the provided list.
    """

    max_retries = 3
    retry_count = 0
    base_delay = 5

    while retry_count <= max_retries:
        try:
            completion = rate_limited_api_call(
                client.beta.chat.completions.parse,
                model="large",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.1,
                response_format=CompaniesExtractor,
            )

            log_api_call(
                system_prompt,
                user_prompt,
                "large",
                completion,
                response_type="companies_extractor",
            )

            result = get_parsed_content(completion, CompaniesExtractor)

            corrected_companies = []
            for company in result.companies:
                if company in all_company_names:
                    corrected_companies.append(company)
                else:
                    corrected = find_closest_company_match(company, all_company_names)
                    if corrected and corrected not in corrected_companies:
                        logger.info(
                            f"Corrected company name from '{company}' to '{corrected}'"
                        )
                        corrected_companies.append(corrected)
                    else:
                        logger.warning(
                            f"Could not find match for company name '{company}'"
                        )

            if not corrected_companies:
                logger.warning(f"No valid companies extracted from question")
            else:
                logger.info(f"Extracted companies: {corrected_companies}")
                logger.info(
                    f"Primary metric: {result.primary_metric}, type: {result.metric_type if result.metric_type else 'unspecified'}"
                )

            return corrected_companies

        except Exception as e:
            error_message = str(e)
            retry_count += 1

            if "Connection error" in error_message and retry_count <= max_retries:
                delay = base_delay * (2 ** (retry_count - 1))

                logger.warning(
                    f"Connection error in extracting companies. Retrying attempt {retry_count}/{max_retries} after {delay} seconds. Error: {error_message}"
                )
                time.sleep(delay)
            else:
                logger.error(f"Error extracting companies with LLM: {error_message}")
                logger.info("Falling back to rule-based company extraction")

                extracted_companies = []

                quoted_pattern = r'"([^"]+)"|\'([^\']+)\''
                quoted_matches = re.findall(quoted_pattern, question_text)

                quoted_companies = []
                for match_tuple in quoted_matches:
                    for match in match_tuple:
                        if match: #скипаю пустую
                            quoted_companies.append(match)

                if quoted_companies:
                    logger.info(
                        f"Found {len(quoted_companies)} companies in quotes: {quoted_companies}"
                    )

                    for company in quoted_companies:
                        if company in all_company_names:
                            extracted_companies.append(company)
                        else:
                            corrected = find_closest_company_match(
                                company, all_company_names
                            )
                            if corrected and corrected not in extracted_companies:
                                extracted_companies.append(corrected)

                if not extracted_companies:
                    for company in all_company_names:
                        if company in question_text:
                            extracted_companies.append(company)

                if not extracted_companies:
                    cap_pattern = r"\b[A-Z][a-zA-Z0-9\.\s,&]*(?:[a-z][a-zA-Z0-9\.\s,&]*)?(?:Inc|Corp|Ltd|LLC|Limited|Group|Holdings|Plc|Co)\b"
                    potential_companies = re.findall(cap_pattern, question_text)

                    for potential in potential_companies:
                        corrected = find_closest_company_match(
                            potential, all_company_names
                        )
                        if corrected and corrected not in extracted_companies:
                            extracted_companies.append(corrected)

                logger.info(
                    f"Extracted companies (fallback method): {extracted_companies}"
                )
                return extracted_companies


def determine_metric_type(question_text: str) -> str:
    logger.info(f"Extracting metric type from question: {question_text}")

    system_prompt = """
    You are an expert financial and business analyst specializing in extracting specific metrics from questions.
    Your task is to identify the primary metric being asked about in a question and classify it into one of the standardized metric types.
    
    # Available Metric Types
    
    ## Financial Metrics
    - total_revenue: Total revenue, sales, income, turnover, or top line
    - total_assets: Total assets on the balance sheet, company's owned resources
    - net_income: Net income, profit, earnings, or bottom line
    - cash_flow_from_operations: Cash flow from operating activities, operational cash flow
    - gross_margin_percentage: Gross margin as a percentage of revenue
    - capital_expenditures: Capital expenditures (CapEx), investments in physical assets
    - dividend_per_share: Dividend paid per share of stock
    - executive_compensation: Executive pay, remuneration, salary, or benefits
    
    ## Operational Metrics
    - employee_count: Number of employees, workforce size, headcount, staff count
    - number_of_facilities: Number of facilities, factories, offices, etc.
    - number_of_stores: Number of stores, retail outlets, branches, locations
    - cloud_storage_capacity: Cloud storage capacity in TB, data storage capacity
    - customer_base: Size of customer base, number of clients, customer count
    - active_user_count: Number of active users, active accounts, active customers
    - fleet_size: Size of vehicle fleet, number of vehicles, transport capacity
    - number_of_patents: Number of patents, intellectual property count
    - tech_staff_headcount: Number of technical staff, engineers, IT personnel
    - rd_spending: R&D spending, research investment, development costs
    
    ## Industry-Specific Metrics
    - number_of_pharmaceutical_patents: Number of pharmaceutical or drug patents
    - number_of_clinics: Number of healthcare clinics or medical facilities
    - number_of_healthcare_professionals: Number of doctors, nurses, medical staff
    - non_performing_loan_ratio: Non-performing loan ratio, bad debt percentage
    - co2_emissions: CO2 emissions, carbon footprint, greenhouse gas emissions
    - number_of_active_software_licenses: Number of active software licenses
    - number_of_active_licensing_deals: Number of active licensing arrangements
    - renewable_energy_percentage: Percentage of renewable energy usage
    - power_generation_capacity: Power generation capacity in MW
    - generic_product_count: Generic product count, number of non-branded products
    - number_of_fulfillment_centers: Number of fulfillment centers, warehouses
    - total_deposits: Total deposits (banking), customer deposits
    - outstanding_insurance_claims: Outstanding insurance claims, unpaid claims
    - e_commerce_active_customers: E-commerce active customers, online shoppers
    - number_of_hotels: Number of hotels, hospitality properties
    - clinical_trial_sites: Number of clinical trial sites, research locations
    - number_of_hybrid_models: Number of hybrid vehicle models
    - year_end_user_base: Year-end user base, user count at fiscal year end
    - net_interest_margin: Net interest margin, interest rate spread
    - market_share: Market share percentage, industry position
    
    # Your Analysis Task
    
    1. Carefully analyze the question to identify what information is being requested
    2. Determine which standardized metric type best matches the information request
    3. If the metric doesn't clearly match any defined type, use "unknown"
    4. Provide detailed reasoning for your selection, including:
       - Key words or phrases in the question that indicate the metric type
       - Any disambiguation between similar metrics
       - Why this metric type is most appropriate for the question
    5. Include a concise description of what this metric measures
    6. Classify the metric into a broader domain (financial, operations, corporate, leadership, other)
    """

    user_prompt = f"""
    Analyze the following question and identify the specific metric type being asked about:
    
    Question: {question_text}
    
    Determine which standardized metric type this question is asking about, explain your reasoning,
    provide a brief description of what this metric measures, and classify it into a broader domain.
    """

    max_retries = 3
    retry_count = 0
    base_delay = 5

    while retry_count <= max_retries:
        try:
            completion = rate_limited_api_call(
                client.beta.chat.completions.parse,
                model="large",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.1,
                response_format=MetricExtractor,
            )

            log_api_call(
                system_prompt,
                user_prompt,
                "large",
                completion,
                response_type="metric_extractor",
            )

            result = get_parsed_content(completion, MetricExtractor)

            logger.info(
                f"Identified metric: {result.metric_type}, domain: {result.domain}"
            )
            logger.info(f"Metric description: {result.metric_description}")

            return result.metric_type

        except Exception as e:
            error_message = str(e)
            retry_count += 1

            if "Connection error" in error_message and retry_count <= max_retries:
                delay = base_delay * (2 ** (retry_count - 1))

                logger.warning(
                    f"Connection error in determining metric type. Retrying attempt {retry_count}/{max_retries} after {delay} seconds. Error: {error_message}"
                )
                time.sleep(delay)
            else:
                logger.error(f"Error extracting metric type with LLM: {error_message}")
                logger.info("Falling back to keyword-based metric detection")

                metric_keywords = {
                    "revenue": "total_revenue",
                    "sales": "total_revenue",
                    "income": "total_revenue",
                    "turnover": "total_revenue",
                    "earnings": "total_revenue",
                    "assets": "total_assets",
                    "balance sheet": "total_assets",
                    "net income": "net_income",
                    "profit": "net_income",
                    "net earnings": "net_income",
                    "bottom line": "net_income",
                    "cash flow": "cash_flow_from_operations",
                    "operating cash": "cash_flow_from_operations",
                    "gross margin": "gross_margin_percentage",
                    "profit margin": "gross_margin_percentage",
                    "margin percentage": "gross_margin_percentage",
                    "capital expenditure": "capital_expenditures",
                    "capex": "capital_expenditures",
                    "capital spending": "capital_expenditures",
                    "dividend": "dividend_per_share",
                    "dividend per share": "dividend_per_share",
                    "dps": "dividend_per_share",
                    "executive compensation": "executive_compensation",
                    "executive pay": "executive_compensation",
                    "ceo salary": "executive_compensation",
                    "executive remuneration": "executive_compensation",
                    "employees": "employee_count",
                    "workforce": "employee_count",
                    "headcount": "employee_count",
                    "staff": "employee_count",
                    "personnel": "employee_count",
                    "employees let go": "employees_let_go",
                    "layoffs": "employees_let_go",
                    "staff reductions": "employees_let_go",
                    "facilities": "number_of_facilities",
                    "factories": "number_of_facilities",
                    "offices": "number_of_facilities",
                    "locations": "number_of_facilities",
                    "stores": "number_of_stores",
                    "retail outlets": "number_of_stores",
                    "branches": "number_of_stores",
                    "shops": "number_of_stores",
                    "cloud storage": "cloud_storage_capacity",
                    "data storage": "cloud_storage_capacity",
                    "storage capacity": "cloud_storage_capacity",
                    "customer base": "customer_base",
                    "clients": "customer_base",
                    "customers": "customer_base",
                    "active users": "active_user_count",
                    "active accounts": "active_user_count",
                    "user base": "active_user_count",
                    "fleet": "fleet_size",
                    "vehicles": "fleet_size",
                    "car fleet": "fleet_size",
                    "truck fleet": "fleet_size",
                    "patents": "number_of_patents",
                    "intellectual property": "number_of_patents",
                    "pharmaceutical patents": "number_of_pharmaceutical_patents",
                    "drug patents": "number_of_pharmaceutical_patents",
                    "clinics": "number_of_clinics",
                    "medical facilities": "number_of_clinics",
                    "healthcare centers": "number_of_clinics",
                    "healthcare professionals": "number_of_healthcare_professionals",
                    "medical staff": "number_of_healthcare_professionals",
                    "doctors": "number_of_healthcare_professionals",
                    "nurses": "number_of_healthcare_professionals",
                    "non-performing loan": "non_performing_loan_ratio",
                    "npl ratio": "non_performing_loan_ratio",
                    "bad debt": "non_performing_loan_ratio",
                    "co2": "co2_emissions",
                    "carbon": "co2_emissions",
                    "emissions": "co2_emissions",
                    "carbon footprint": "co2_emissions",
                    "greenhouse gas": "co2_emissions",
                    "software licenses": "number_of_active_software_licenses",
                    "active licenses": "number_of_active_software_licenses",
                    "licensing deals": "number_of_active_licensing_deals",
                    "licensing agreements": "number_of_active_licensing_deals",
                    "tech staff": "tech_staff_headcount",
                    "it personnel": "tech_staff_headcount",
                    "engineers": "tech_staff_headcount",
                    "developers": "tech_staff_headcount",
                    "renewable energy": "renewable_energy_percentage",
                    "clean energy": "renewable_energy_percentage",
                    "green energy": "renewable_energy_percentage",
                    "power generation": "power_generation_capacity",
                    "electricity generation": "power_generation_capacity",
                    "generation capacity": "power_generation_capacity",
                    "mw capacity": "power_generation_capacity",
                    "generic products": "generic_product_count",
                    "generics": "generic_product_count",
                    "r&d": "rd_spending",
                    "research and development": "rd_spending",
                    "research spending": "rd_spending",
                    "fulfillment centers": "number_of_fulfillment_centers",
                    "distribution centers": "number_of_fulfillment_centers",
                    "warehouses": "number_of_fulfillment_centers",
                    "deposits": "total_deposits",
                    "bank deposits": "total_deposits",
                    "customer deposits": "total_deposits",
                    "insurance claims": "outstanding_insurance_claims",
                    "claims": "outstanding_insurance_claims",
                    "e-commerce customers": "e_commerce_active_customers",
                    "online customers": "e_commerce_active_customers",
                    "online shoppers": "e_commerce_active_customers",
                    "hotels": "number_of_hotels",
                    "hotel properties": "number_of_hotels",
                    "clinical trials": "clinical_trial_sites",
                    "trial sites": "clinical_trial_sites",
                    "hybrid models": "number_of_hybrid_models",
                    "hybrid vehicles": "number_of_hybrid_models",
                    "year-end user": "year_end_user_base",
                    "year end customers": "year_end_user_base",
                    "interest margin": "net_interest_margin",
                    "nim": "net_interest_margin",
                    "market share": "market_share",
                    "industry share": "market_share",
                    "market position": "market_share",
                }

                question_lower = question_text.lower()
                found_metric = None
                max_length = 0

                for keyword, metric in metric_keywords.items():
                    if keyword.lower() in question_lower:
                        if len(keyword) > max_length:
                            max_length = len(keyword)
                            found_metric = metric

                if found_metric:
                    logger.info(f"Identified metric (fallback): {found_metric}")
                    return found_metric

                logger.info("Metric type not identified (fallback), using 'unknown'")
                return "unknown"


def determine_domain_for_metric(metric_type: str) -> str:
    logger.info(f"Determining business domain for metric type: {metric_type}")

    system_prompt = """
    You are an expert financial and business analyst specializing in categorizing business metrics.
    Your task is to determine which domain a specific business metric belongs to.
    
    # Available Business Domains
    
    1. FINANCIAL
       - Definition: Metrics related to monetary performance, financial statements, and economic data
       - Examples: revenue, profit, cash flow, assets, liabilities, margins, dividends, investments
       - Typically found in: Income Statements, Balance Sheets, Cash Flow Statements, Financial Notes
    
    2. OPERATIONS
       - Definition: Metrics related to business activities, resources, infrastructure, and capacity
       - Examples: employees, facilities, production, inventory, efficiency, equipment, capacity
       - Typically found in: Operations Review, Management Discussion, Business Segments
    
    3. CORPORATE
       - Definition: Metrics related to company-wide strategic actions and organizational decisions
       - Examples: mergers, acquisitions, restructuring, policy changes, new ventures, partnerships
       - Typically found in: Chairman's Letter, CEO's Message, Strategic Overview
    
    4. LEADERSHIP
       - Definition: Metrics related to executive management, governance, and organizational structure
       - Examples: leadership changes, executive appointments, board composition, management teams
       - Typically found in: Corporate Governance, Leadership Profiles, Board Information
    
    5. OTHER
       - Definition: Metrics that don't clearly fit into the above categories
       - Examples: industry-specific metrics, specialized indicators, hybrid metrics
       - Typically found in: Various sections specific to the company or industry
    
    # Domain Classification Task
    
    For the given metric type:
    1. Analyze the nature of the metric and what aspect of the business it measures
    2. Determine which domain best encompasses this type of measurement
    3. Provide detailed reasoning explaining why this domain is most appropriate
    4. Include a comprehensive description of the domain's scope and boundaries
    5. Identify the primary focus or central concern that this metric addresses
    
    Be precise in your classification, considering both the direct nature of the metric
    and its typical use and interpretation in business analysis.
    """

    user_prompt = f"""
    Determine which business domain the following metric type belongs to:
    
    Metric type: {metric_type}
    
    Classify this metric into one of these domains:
    - financial
    - operations
    - corporate
    - leadership
    - other
    
    Provide detailed reasoning for your domain classification, including a description
    of the domain and the primary focus this metric addresses within that domain.
    """

    max_retries = 3
    retry_count = 0
    base_delay = 5

    while retry_count <= max_retries:
        try:
            completion = rate_limited_api_call(
                client.beta.chat.completions.parse,
                model="large",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.1,
                response_format=DomainClassifier,
            )

            log_api_call(
                system_prompt,
                user_prompt,
                "large",
                completion,
                response_type="domain_classifier",
            )

            result = get_parsed_content(completion, DomainClassifier)

            logger.info(f"Determined domain: {result.domain} for metric: {metric_type}")
            logger.info(f"Domain description: {result.domain_description}")
            logger.info(f"Primary focus: {result.primary_focus}")

            return result.domain

        except Exception as e:
            error_message = str(e)
            retry_count += 1

            if "Connection error" in error_message and retry_count <= max_retries:
                delay = base_delay * (2 ** (retry_count - 1))

                logger.warning(
                    f"Connection error in domain determination. Retrying attempt {retry_count}/{max_retries} after {delay} seconds. Error: {error_message}"
                )
                time.sleep(delay)
            else:
                logger.error(f"Error determining domain with LLM: {error_message}")
                logger.info("Falling back to predefined domain mapping")

                domain_mapping = {
                    "total_revenue": "financial",
                    "total_assets": "financial",
                    "net_income": "financial",
                    "cash_flow_from_operations": "financial",
                    "gross_margin_percentage": "financial",
                    "capital_expenditures": "financial",
                    "dividend_per_share": "financial",
                    "executive_compensation": "financial",
                    "employee_count": "operations",
                    "employees_let_go": "operations",
                    "number_of_facilities": "operations",
                    "number_of_stores": "operations",
                    "cloud_storage_capacity": "operations",
                    "customer_base": "operations",
                    "active_user_count": "operations",
                    "fleet_size": "operations",
                    "number_of_patents": "operations",
                    "number_of_pharmaceutical_patents": "operations",
                    "number_of_clinics": "operations",
                    "number_of_healthcare_professionals": "operations",
                    "non_performing_loan_ratio": "operations",
                    "co2_emissions": "operations",
                    "number_of_active_software_licenses": "operations",
                    "number_of_active_licensing_deals": "operations",
                    "tech_staff_headcount": "operations",
                    "renewable_energy_percentage": "operations",
                    "total_headcount": "operations",
                    "power_generation_capacity": "operations",
                    "generic_product_count": "operations",
                    "rd_spending": "operations",
                    "number_of_fulfillment_centers": "operations",
                    "total_deposits": "operations",
                    "outstanding_insurance_claims": "operations",
                    "e_commerce_active_customers": "operations",
                    "number_of_hotels": "operations",
                    "clinical_trial_sites": "operations",
                    "number_of_hybrid_models": "operations",
                    "year_end_user_base": "operations",
                    "net_interest_margin": "operations",
                    "market_share": "operations",
                    "mergers_acquisitions": "corporate",
                    "share_buyback_plan": "corporate",
                    "dividend_policy_changes": "corporate",
                    "capital_structure_changes": "corporate",
                    "restructuring_plans": "corporate",
                    "new_product_launches": "corporate",
                    "esg_initiatives": "corporate",
                    "ongoing_litigation": "corporate",
                    "leadership_positions_changed": "leadership",
                    "executive_appointments": "leadership",
                    "executive_departures": "leadership",
                    "board_changes": "leadership",
                    "organizational_restructuring": "leadership",
                }

                domain = domain_mapping.get(metric_type, "other")
                logger.info(
                    f"Determined domain (fallback): {domain} for metric: {metric_type}"
                )
                return domain


def identify_company_and_domain(
    question_text: str, question_kind: str, company_names: List[str]
) -> CompanyIdentifier:
    system_prompt = """
    You are an expert analyst specializing in the analysis of company annual reports.
    Your task is to identify the main company mentioned in the question, the domain of the question,
    and generate THREE DISTINCTLY DIFFERENT query expansions optimized for searching annual reports.
    
    # Question Domains
    
    - financial: Questions about financial metrics, performance, statements (revenue, profit, cash flows, margin, assets)
    - operations: Questions about business operations, capacity, resources (employees, facilities, stores, capacity)
    - corporate: Questions about strategic decisions and company actions (mergers, acquisitions, buybacks, dividends)
    - leadership: Questions about management and executives (leadership changes, appointments, departures)
    - other: Questions that don't clearly fit into the above categories
    
    # Query Expansion Strategy
    
    For each question, generate THREE distinctly different query approaches:
    
    1. QUESTION RESTATEMENT:
       - Create a clear, concise restatement of the original question that resolves ambiguities
       - Remove unnecessary words while preserving the core information need
       - Format as a complete, grammatical question or statement
       - Example: "What was Apple's annual revenue in 2022?" → "Apple annual revenue 2022"
    
    2. QUERY EXPANSION 1 - CORE TERMINOLOGY:
            - Extract ONLY the essential subject matter from the original question
            - Remove ALL company names and verbs
            - Focus on the specific financial metric, operational fact, or corporate information being sought
            - Examples:
            * Original: "What was Microsoft's total revenue in 2022?" → Query 1: "total revenue"
            * Original: "How many employees did Amazon have?" → Query 1: "employees headcount"
            * Original: "Did Apple announce new product launches?" → Query 1: "new product"

    
    3. QUERY EXPANSION 2 - ALTERNATIVE PHRASING:
       - Provide alternative ways to express the same concept using synonyms and related terms
       - Include both formal and informal business language variations
       - Capture different ways the information might be documented across companies
       - Example: "sales income turnover financial performance yearly results"
    
    4. QUERY EXPANSION 3 - CONTEXTUAL INDICATORS:
       - Focus on document structure and context markers that often surround the requested information
       - Include section titles, paragraph starters, and report-specific phrases
       - Add terms that help locate the specific part of the report where this information typically appears
       - Example: "management discussion analysis year ended notes to consolidated compared to previous"
    
    # Your Task
    
    For the provided question:
    
    1. Identify the main company being asked about (must match exactly from the provided list)
    2. List any related companies mentioned in the question (if any)
    3. Determine the primary domain of the question
    4. Generate three query expansions following the strategy above
    5. Provide detailed reasoning for your company and domain identification
    6. Explain your query expansion strategy and how it will help find the answer
    """

    user_prompt = f"""
    Identify the company, domain, and generate optimized search query expansions for the following question:
    
    Question: {question_text}
    Answer type: {question_kind}
    
    List of possible companies:
    {", ".join(company_names[:100])}... (truncated for brevity)
    
    Your task:
    1. Identify the main company this question is asking about (must exactly match a name from the list)
    2. Determine the primary domain of the question (financial, operations, corporate, leadership, other)
    3. Create a clear restatement of the question that resolves ambiguities
    4. Generate three different query expansions using different vocabulary and phrasing approaches
    5. Provide detailed reasoning for your analysis
    """

    logger.info(
        f"Identifying company, domain and generating query expansions for: {question_text}"
    )

    max_retries = 3
    retry_count = 0
    base_delay = 5

    while retry_count <= max_retries:
        try:
            completion = rate_limited_api_call(
                client.beta.chat.completions.parse,
                model="large",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.2,
                response_format=CompanyIdentifier,
            )

            log_api_call(
                system_prompt,
                user_prompt,
                "large",
                completion,
                response_type="company_identifier",
            )

            result = get_parsed_content(completion, CompanyIdentifier)

            if result.company and result.company not in company_names:
                corrected_company = find_closest_company_match(
                    result.company, company_names
                )
                if corrected_company:
                    logger.info(
                        f"Corrected company name from '{result.company}' to '{corrected_company}'"
                    )
                    result.company = corrected_company
                else:
                    logger.warning(
                        f"Could not find match for company name '{result.company}'"
                    )

            if result.related_companies:
                corrected_related = []
                for company in result.related_companies:
                    if company not in company_names:
                        corrected = find_closest_company_match(company, company_names)
                        if corrected:
                            corrected_related.append(corrected)
                    else:
                        corrected_related.append(company)
                result.related_companies = (
                    corrected_related if corrected_related else None
                )

            logger.info(
                f"Identified company: {result.company}, domain: {result.domain}"
            )
            logger.info(f"Query expansions generated:")
            logger.info(f"1. Rephrased question: {result.rephrased_question}")
            logger.info(f"2. Core terminology: {result.query_expansion_1}")
            logger.info(f"3. Alternative phrasing: {result.query_expansion_2}")
            logger.info(f"4. Contextual indicators: {result.query_expansion_3}")

            return result

        except Exception as e:
            error_message = str(e)
            retry_count += 1

            if "Connection error" in error_message and retry_count <= max_retries:
                delay = base_delay * (2 ** (retry_count - 1))

                logger.warning(
                    f"Connection error in company/domain identification. Retrying attempt {retry_count}/{max_retries} after {delay} seconds. Error: {error_message}"
                )
                time.sleep(delay)
            else:
                logger.error(
                    f"Error identifying company and domain with LLM: {error_message}"
                )
                logger.info(
                    "Falling back to rule-based company and domain identification"
                )

                found_company = None
                for company in company_names:
                    if company in question_text:
                        found_company = company
                        break

                if not found_company:
                    potential_companies = re.findall(
                        r"\b[A-Z][a-zA-Z0-9]*\b", question_text
                    )
                    for potential in potential_companies:
                        if len(potential) >= 4:
                            found_company = find_closest_company_match(
                                potential, company_names
                            )
                            if found_company:
                                break

                metric_type = determine_metric_type(question_text)
                domain = (
                    determine_domain_for_metric(metric_type)
                    if metric_type != "unknown"
                    else "other"
                )

                keywords = re.findall(
                    r"\b[a-zA-Z][a-zA-Z0-9]+\b", question_text.lower()
                )
                keywords = [
                    word
                    for word in keywords
                    if word
                    not in [
                        "what",
                        "which",
                        "when",
                        "where",
                        "how",
                        "did",
                        "does",
                        "is",
                        "are",
                        "was",
                        "were",
                        "the",
                        "and",
                        "or",
                        "for",
                        "to",
                        "in",
                        "on",
                        "at",
                        "by",
                        "from",
                    ]
                ]

                domain_terms = {
                    "financial": {
                        "core": [
                            "revenue",
                            "profit",
                            "earnings",
                            "income",
                            "financial results",
                            "fiscal year",
                        ],
                        "alt": [
                            "sales",
                            "turnover",
                            "bottom line",
                            "net profit",
                            "financial performance",
                        ],
                        "context": [
                            "annual report",
                            "financial statements",
                            "balance sheet",
                            "income statement",
                            "consolidated",
                        ],
                    },
                    "operations": {
                        "core": [
                            "employees",
                            "workforce",
                            "headcount",
                            "facilities",
                            "capacity",
                        ],
                        "alt": [
                            "staff",
                            "personnel",
                            "infrastructure",
                            "operations",
                            "resources",
                        ],
                        "context": [
                            "operational review",
                            "business segments",
                            "corporate overview",
                            "company profile",
                        ],
                    },
                    "corporate": {
                        "core": [
                            "merger",
                            "acquisition",
                            "buyback",
                            "dividend",
                            "restructuring",
                        ],
                        "alt": [
                            "strategic initiative",
                            "corporate action",
                            "company decision",
                            "board approval",
                        ],
                        "context": [
                            "corporate developments",
                            "strategic overview",
                            "shareholder letter",
                            "ceo statement",
                        ],
                    },
                    "leadership": {
                        "core": [
                            "executives",
                            "management",
                            "leadership",
                            "board members",
                            "director",
                        ],
                        "alt": [
                            "ceo",
                            "cfo",
                            "executive team",
                            "c-suite",
                            "management team",
                        ],
                        "context": [
                            "leadership profiles",
                            "corporate governance",
                            "management discussion",
                            "board of directors",
                        ],
                    },
                    "other": {
                        "core": [
                            "information",
                            "details",
                            "report",
                            "announcement",
                            "disclosure",
                        ],
                        "alt": [
                            "statement",
                            "communication",
                            "release",
                            "update",
                            "summary",
                        ],
                        "context": [
                            "annual report",
                            "company overview",
                            "management letter",
                            "business review",
                        ],
                    },
                }

                rephrased_question = " ".join(
                    [word for word in keywords if len(word) > 3]
                )
                if found_company:
                    rephrased_question = f"{found_company} {rephrased_question}"

                domain_core = domain_terms.get(domain, domain_terms["other"])["core"]
                domain_alt = domain_terms.get(domain, domain_terms["other"])["alt"]
                domain_context = domain_terms.get(domain, domain_terms["other"])[
                    "context"
                ]

                query_expansion_1 = (
                    " ".join(random.sample(domain_core, min(3, len(domain_core))))
                    + " "
                    + rephrased_question
                )
                query_expansion_2 = (
                    " ".join(random.sample(domain_alt, min(3, len(domain_alt))))
                    + " "
                    + rephrased_question
                )
                query_expansion_3 = (
                    " ".join(random.sample(domain_context, min(3, len(domain_context))))
                    + " "
                    + rephrased_question
                )

                logger.info(
                    f"Fallback identification: company: {found_company}, domain: {domain}"
                )
                logger.info(f"Generated query expansions (fallback):")
                logger.info(f"1. Rephrased: {rephrased_question}")
                logger.info(f"2. Core: {query_expansion_1}")
                logger.info(f"3. Alt: {query_expansion_2}")
                logger.info(f"4. Context: {query_expansion_3}")

                return CompanyIdentifier(
                    company=found_company,
                    related_companies=None,
                    domain=domain,
                    question_user=question_text,
                    rephrased_question=rephrased_question,
                    query_expansion_1=query_expansion_1,
                    query_expansion_2=query_expansion_2,
                    query_expansion_3=query_expansion_3,
                    reasoning=f"Company and domain identified by fallback mechanism due to LLM error.",
                    reasoning_about_query="Query expansions generated by fallback mechanism to maximize search coverage using domain-specific terminology.",
                    reasoning_step2="These query variations target different ways the information might be expressed in annual reports by using domain-specific vocabulary.",
                )


def extract_financial_data_from_page(
    page_text: str, page_num: int, company_name: str
) -> FinancialMetricsData:
    system_prompt = """
    You are a financial analyst specializing in extracting financial metrics from company annual reports.
    Your task is to carefully analyze the text of an annual report page and extract all available financial metrics.
    
    # Financial Metrics to Extract
    
    Focus on extracting these key financial metrics:
    
    1. Total Revenue
       - Definition: Total income generated from all business activities
       - Also known as: sales, turnover, gross revenue, top line
       - Example formats: "$10.5 billion", "£7.2 million", "€840 million", "10,500,000 USD"
    
    2. Cash Flow from Operations
       - Definition: Net cash generated from regular business operations
       - Also known as: operating cash flow, cash from operating activities
       - Example: "$1.2 billion positive cash flow from operations"
    
    3. Gross Margin Percentage
       - Definition: Gross profit as a percentage of revenue
       - Also known as: gross profit margin, gross margin ratio
       - Example: "Gross margin increased to 42.5%"
    
    4. Net Income
       - Definition: Total earnings after all expenses, taxes and costs
       - Also known as: net profit, net earnings, bottom line
       - Example: "Net income of $750 million, up 8% year-over-year"
    
    5. Total Assets
       - Definition: Combined value of all company-owned resources
       - Look for: Balance sheet items, references to combined/total assets
       - Example: "Total assets reached $15.3 billion at fiscal year end"
    
    6. Capital Expenditures
       - Definition: Funds used to acquire, upgrade, or maintain physical assets
       - Also known as: CapEx, capital spending, fixed asset investment
       - Example: "Capital expenditures for the year were $2.1 billion"
    
    7. Dividend per Share
       - Definition: Amount of dividends paid per outstanding share
       - Also known as: DPS, per-share dividend, dividend rate
       - Example: "The company declared a dividend of $0.88 per share"
    
    8. Executive Compensation
       - Definition: Total remuneration provided to executives
       - Also known as: executive pay, management compensation
       - Example: "CEO total compensation package of $12.5 million"
    
    # Extraction Guidelines
    
    1. EXACT VALUES:
       - Extract precise numerical values for each metric
       - Include the correct units (millions, billions) and currency (USD, EUR, GBP)
       - If a range is given, note both values or use the most recent
    
    2. CONTEXT AWARENESS:
       - Determine if metrics are for the current fiscal year, previous year, or projections
       - Note whether values represent increases, decreases, or stable figures
       - Pay attention to comparative language ("up from", "decreased by")
    
    3. CURRENCY HANDLING:
       - Always record the currency unit (USD, EUR, GBP, etc.)
       - If currency is not specified but can be inferred from context, note your inference
       - If multiple currencies appear, prioritize the company's primary reporting currency
    
    4. EVIDENCE CITATION:
       - Support each extraction with direct quotes from the text
       - Note the specific context surrounding each metric
       - Explain your confidence level for each extraction
    
    5. DATA GAPS:
       - If information for a specific metric is not found, leave that field empty
       - Do not guess or infer values that aren't clearly stated or calculable
       - Only extract data that appears directly on this specific page
    
    # Output Requirements
    
    1. Set found_data=True ONLY if you find at least one concrete financial metric on the page
    2. For each extracted metric, provide the specific value and supporting evidence
    3. Include detailed reasoning explaining how you identified and validated each metric
    4. Add a second level of reasoning that interprets these metrics in a business context
    5. For fields where data is not found, leave them as null/None
    """

    user_prompt = f"""
    Analyze the following page {page_num} from {company_name}'s annual report and extract all financial metrics.
    
    PAGE TEXT:
    {page_text}
    
    For each financial metric found on this page:
    1. Extract the precise numerical value with correct units and currency
    2. Provide direct quotes from the text supporting your extraction
    3. Explain your confidence level and any contextual considerations
    4. Add business context interpretation for the metrics found
    
    Set found_data=True only if you find at least one concrete financial metric on this page.
    """

    logger.info(
        f"Extracting financial information from page {page_num} for {company_name}"
    )

    max_retries = 3
    retry_count = 0
    base_delay = 5

    while retry_count <= max_retries:
        try:
            completion = rate_limited_api_call(
                client.beta.chat.completions.parse,
                model="large",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.1,
                response_format=FinancialMetricsData,
            )

            log_api_call(
                system_prompt,
                user_prompt,
                "large",
                completion,
                response_type="financial_data",
                page_num=page_num,
                company_name=company_name,
            )

            result = get_parsed_content(completion, FinancialMetricsData)

            result.page_num = page_num

            if result.found_data:
                logger.info(
                    f"Financial data found on page {page_num} for {company_name}"
                )

                metrics_found = []
                if result.total_revenue is not None:
                    metrics_found.append(
                        f"total_revenue: {result.total_revenue} {result.currency or 'unknown currency'}"
                    )
                if result.net_income is not None:
                    metrics_found.append(
                        f"net_income: {result.net_income} {result.currency or 'unknown currency'}"
                    )
                if result.total_assets is not None:
                    metrics_found.append(
                        f"total_assets: {result.total_assets} {result.currency or 'unknown currency'}"
                    )
                if result.cash_flow_from_operations is not None:
                    metrics_found.append(
                        f"cash_flow: {result.cash_flow_from_operations} {result.currency or 'unknown currency'}"
                    )
                if result.gross_margin_percentage is not None:
                    metrics_found.append(
                        f"gross_margin: {result.gross_margin_percentage}%"
                    )
                if result.capital_expenditures is not None:
                    metrics_found.append(
                        f"capex: {result.capital_expenditures} {result.currency or 'unknown currency'}"
                    )
                if result.dividend_per_share is not None:
                    metrics_found.append(
                        f"dividend: {result.dividend_per_share} {result.currency or 'unknown currency'}/share"
                    )
                if result.executive_compensation is not None:
                    metrics_found.append(
                        f"exec_comp: {result.executive_compensation} {result.currency or 'unknown currency'}"
                    )

                logger.info(f"Metrics found: {', '.join(metrics_found)}")
            else:
                logger.info(
                    f"No financial data found on page {page_num} for {company_name}"
                )

            return result

        except Exception as e:
            error_message = str(e)
            retry_count += 1

            if "Connection error" in error_message and retry_count <= max_retries:
                delay = base_delay * (2 ** (retry_count - 1))

                logger.warning(
                    f"Connection error in financial data extraction. Retrying attempt {retry_count}/{max_retries} after {delay} seconds. Error: {error_message}"
                )
                time.sleep(delay)
            else:
                logger.error(
                    f"Error extracting financial information from page {page_num}: {error_message}"
                )

                return FinancialMetricsData(
                    page_num=page_num,
                    found_data=False,
                    reasoning=f"Failed to extract financial information due to: {error_message}",
                    reasoning_step2="No business context analysis possible due to extraction failure.",
                )


def extract_business_operations_from_page(
    page_text: str, page_num: int, company_name: str
) -> BusinessOperationsData:
    system_prompt = """
    You are a business analyst specializing in extracting operational metrics from company annual reports.
    Your task is to carefully analyze the text of an annual report page and extract all available information
    about the company's business operations.
    
    # Operational Metrics to Extract
    
    Focus on extracting these key operational metrics:
    
    ## Workforce Metrics
    1. Employee Count
       - Definition: Total number of employees working for the company
       - Also known as: headcount, workforce size, staff count, personnel
       - Example: "Our global workforce reached 85,000 employees"
    
    2. Employees Let Go
       - Definition: Number of employees terminated or laid off during the period
       - Also known as: layoffs, workforce reduction, downsizing
       - Example: "The restructuring resulted in 1,200 positions being eliminated"
    
    3. Tech Staff Headcount
       - Definition: Number of employees in technology-related roles
       - Also known as: IT staff, technical personnel, engineering headcount
       - Example: "The company employs 5,300 software engineers and IT professionals"
    
    4. Total Headcount
       - Definition: Comprehensive count of all people employed by the company
       - Look for: References to global workforce, total personnel, worldwide employees
       - Example: "Total headcount grew to 42,500 across all regions"
    
    ## Infrastructure Metrics
    5. Number of Facilities
       - Definition: Total count of physical locations operated by the company
       - Also known as: locations, sites, offices, plants, factories
       - Example: "The company operates 127 facilities across 30 countries"
    
    6. Number of Stores
       - Definition: Count of retail locations or storefronts
       - Also known as: retail outlets, branches, shops, locations
       - Example: "Our retail footprint expanded to 850 stores worldwide"
    
    7. Cloud Storage Capacity
       - Definition: Company's cloud storage capacity measured in terabytes (TB)
       - Also known as: data storage capacity, digital storage infrastructure
       - Example: "Cloud storage capacity increased to 15.2 petabytes"
    
    8. Number of Fulfillment Centers
       - Definition: Count of warehouses or distribution centers for order processing
       - Also known as: distribution centers, logistics hubs, warehouses
       - Example: "The company added 5 new fulfillment centers, bringing the total to 28"
    
    ## Customer and Market Metrics
    9. Customer Base
       - Definition: Total number of customers currently served by the company
       - Also known as: client base, customer count, active accounts
       - Example: "Our customer base grew to 25 million active accounts"
    
    10. Active User Count
        - Definition: Number of users actively engaging with products/services
        - Also known as: active accounts, engaged users, active customers
        - Example: "Monthly active users reached 120 million in Q4"
    
    11. E-commerce Active Customers
        - Definition: Number of customers who made online purchases in the period
        - Also known as: online shoppers, digital customers, e-commerce users
        - Example: "E-commerce active customers grew by 15% to 8.5 million"
    
    12. Year-End User Base
        - Definition: Total number of users at the end of the fiscal year
        - Look for: Year-end statistics, fiscal year-end counts, closing period numbers
        - Example: "Year-end user base totaled 45.3 million, up 12% year-over-year"
    
    13. Market Share
        - Definition: Company's percentage share of the total market for its products
        - Also known as: industry share, market position, competitive standing
        - Example: "Market share in the premium segment increased to 32.5%"
    
    ## Industry-Specific Metrics
    
    [Additional 17 specialized metrics including fleet size, patents, clinics, etc.]
    
    # Extraction Guidelines
    
    1. PRECISE VALUES:
       - Extract exact numerical values for each metric
       - Include appropriate units (count, percentage, TB, MW, etc.)
       - If ranges or multiple values are given, prioritize the most recent or comprehensive figure
    
    2. CONTEXT AWARENESS:
       - Determine if metrics represent current values, year-over-year changes, or projections
       - Pay attention to time periods (e.g., "as of fiscal year end", "during Q4")
       - Note whether values represent increases, decreases, or stable figures
    
    3. EVIDENCE CITATION:
       - Support each extraction with direct quotes from the text
       - Note the specific context surrounding each metric
       - Explain your confidence level for each extraction
    
    4. BUSINESS IMPLICATIONS:
       - For each significant metric found, provide brief analysis of its business implications
       - Consider how these operational metrics reflect company strategy and performance
       - Note any unusual patterns or significant changes in the metrics
    
    5. DATA GAPS:
       - If information for a specific metric is not found, leave that field empty
       - Do not guess or infer values that aren't clearly stated or calculable
       - Only extract data that appears directly on this specific page
    
    # Output Requirements
    
    1. Set found_data=True ONLY if you find at least one concrete operational metric on the page
    2. For each extracted metric, provide the specific value and supporting evidence
    3. Include detailed reasoning explaining how you identified and validated each metric
    4. Add a second level of reasoning that interprets these metrics in a business context
    5. For fields where data is not found, leave them as null/None
    """

    user_prompt = f"""
    Analyze the following page {page_num} from {company_name}'s annual report and extract all business operations metrics.
    
    PAGE TEXT:
    {page_text}
    
    For each operational metric found on this page:
    1. Extract the precise numerical value with correct units
    2. Provide direct quotes from the text supporting your extraction
    3. Explain your confidence level and any contextual considerations
    4. Add business implications analysis for the metrics found
    
    Set found_data=True only if you find at least one concrete operational metric on this page.
    """

    logger.info(
        f"Extracting business operations data from page {page_num} for {company_name}"
    )

    max_retries = 3
    retry_count = 0
    base_delay = 5

    while retry_count <= max_retries:
        try:
            completion = rate_limited_api_call(
                client.beta.chat.completions.parse,
                model="large",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.1,
                response_format=BusinessOperationsData,
            )

            log_api_call(
                system_prompt,
                user_prompt,
                "large",
                completion,
                response_type="business_operations",
                page_num=page_num,
                company_name=company_name,
            )

            result = get_parsed_content(completion, BusinessOperationsData)

            result.page_num = page_num

            if result.found_data:
                logger.info(
                    f"Business operations data found on page {page_num} for {company_name}"
                )

                metrics_found = []
                if result.employee_count is not None:
                    metrics_found.append(f"employee_count: {result.employee_count}")
                if result.employees_let_go is not None:
                    metrics_found.append(f"employees_let_go: {result.employees_let_go}")
                if result.number_of_facilities is not None:
                    metrics_found.append(f"facilities: {result.number_of_facilities}")
                if result.number_of_stores is not None:
                    metrics_found.append(f"stores: {result.number_of_stores}")
                if result.cloud_storage_capacity is not None:
                    metrics_found.append(
                        f"cloud_storage: {result.cloud_storage_capacity} TB"
                    )
                if result.customer_base is not None:
                    metrics_found.append(f"customers: {result.customer_base}")
                if result.market_share is not None:
                    metrics_found.append(f"market_share: {result.market_share}%")
                if result.total_headcount is not None:
                    metrics_found.append(f"total_headcount: {result.total_headcount}")

                logger.info(f"Metrics found: {', '.join(metrics_found)}")
            else:
                logger.info(
                    f"No business operations data found on page {page_num} for {company_name}"
                )

            return result

        except Exception as e:
            error_message = str(e)
            retry_count += 1

            if "Connection error" in error_message and retry_count <= max_retries:
                delay = base_delay * (2 ** (retry_count - 1))

                logger.warning(
                    f"Connection error in business operations extraction. Retrying attempt {retry_count}/{max_retries} after {delay} seconds. Error: {error_message}"
                )
                time.sleep(delay)
            else:
                logger.error(
                    f"Error extracting business operations data from page {page_num}: {error_message}"
                )

                return BusinessOperationsData(
                    page_num=page_num,
                    found_data=False,
                    reasoning=f"Failed to extract business operations data due to: {error_message}",
                    reasoning_step2="No operational analysis possible due to extraction failure.",
                )


def extract_corporate_actions_from_page(
    page_text: str, page_num: int, company_name: str
) -> CorporateActionsData:
    system_prompt = """
    You are a corporate strategy expert specializing in analyzing company annual reports.
    Your task is to carefully analyze the text of an annual report page and extract all available
    information about corporate actions and strategic initiatives.
    
    # Corporate Actions to Extract
    
    Focus on extracting these key corporate actions:
    
    ## Strategic Corporate Moves
    1. Mergers and Acquisitions
       - Definition: Combinations with or purchases of other companies/businesses
       - Look for: Discussion of M&A activity, acquired companies, merger agreements
       - Example: "The company completed the acquisition of XYZ Corp for $2.3 billion"
       - Record: Boolean (true/false) + Details list with names, values, status
    
    2. Share Buyback Plans
       - Definition: Company programs to repurchase its own shares from the market
       - Also known as: stock repurchase, share repurchase program
       - Example: "The Board authorized a $5 billion share repurchase program"
       - Record: Boolean (true/false)
    
    3. Dividend Policy Changes
       - Definition: Modifications to how the company distributes profits to shareholders
       - Look for: Dividend increases/decreases, policy revisions, special dividends
       - Example: "The dividend policy was revised to target a 40% payout ratio"
       - Record: Boolean (true/false)
    
    4. Capital Structure Changes
       - Definition: Alterations to the company's debt-to-equity makeup
       - Look for: Refinancing, debt issuance, equity offerings, capital reorganization
       - Example: "The company issued $1.2 billion in senior notes to reduce short-term debt"
       - Record: Boolean (true/false)
    
    5. Restructuring Plans
       - Definition: Significant reorganization of business operations or corporate structure
       - Look for: Business unit reorganizations, major cost-cutting initiatives
       - Example: "The restructuring plan is expected to generate $300M in annual savings"
       - Record: Boolean (true/false)
    
    ## Product and Business Development
    6. New Product Launches
       - Definition: Introduction of new products or services to the market
       - Look for: Product announcements, new offerings, market introductions
       - Example: "The company launched its new flagship smartphone in September"
       - Record: Boolean (true/false) + Product names list + Last product launched
    
    ## Sustainability and Legal
    7. ESG Initiatives
       - Definition: Environmental, Social, and Governance programs or commitments
       - Look for: Sustainability targets, social responsibility initiatives, governance changes
       - Example: "The company committed to carbon neutrality by 2030"
       - Record: Boolean (true/false) + Initiative details list
    
    8. Ongoing Litigation
       - Definition: Legal proceedings involving the company
       - Look for: Lawsuits, regulatory investigations, legal settlements
       - Example: "The company settled the patent infringement case for $75 million"
       - Record: Boolean (true/false) + Litigation details list
    
    # Extraction Guidelines
    
    1. BINARY IDENTIFICATION:
       - For each category, determine if there is clear evidence of the action (true/false)
       - Only mark as true if the page explicitly discusses the action
       - Pay attention to timeframe (current reporting period vs. historical references)
    
    2. DETAILED EXTRACTION:
       - For actions marked true, extract specific details when available
       - Include names, monetary values, dates, and status information
       - For product launches, identify specific product names and the most recent launch
    
    3. CONTEXT AWARENESS:
       - Differentiate between completed actions, ongoing initiatives, and future plans
       - Note whether actions represent new announcements or updates to previous initiatives
       - Consider the significance and scale of the actions described
    
    4. EVIDENCE CITATION:
       - Support each extraction with direct quotes from the text
       - Note the specific context surrounding each action
       - Explain your confidence level for each extraction
    
    5. STRATEGIC ANALYSIS:
       - For each significant action found, provide brief analysis of its strategic implications
       - Consider how these actions align with company strategy and market positioning
       - Analyze potential impacts on stakeholders (investors, employees, customers)
    
    6. DATA GAPS:
       - If information for a specific action is not found, leave that field as false or empty
       - Do not guess or infer actions that aren't clearly stated
       - Only extract information that appears directly on this specific page
    
    # Output Requirements
    
    1. Set found_data=True ONLY if you find evidence of at least one corporate action on the page
    2. For each action identified, provide the boolean value and any detailed information
    3. Include detailed reasoning explaining how you identified each action
    4. Add a second level of reasoning that analyzes the strategic implications
    5. For actions not found on the page, set the corresponding fields to false or null/None
    """

    user_prompt = f"""
    Analyze the following page {page_num} from {company_name}'s annual report and extract all information about corporate actions.
    
    PAGE TEXT:
    {page_text}
    
    For each corporate action found on this page:
    1. Determine if there is clear evidence of the action (true/false)
    2. Extract specific details when available (names, values, dates, status)
    3. Provide direct quotes from the text supporting your extraction
    4. Add strategic analysis of the implications of each action
    
    Set found_data=True only if you find evidence of at least one corporate action on this page.
    """

    logger.info(
        f"Extracting corporate actions data from page {page_num} for {company_name}"
    )

    max_retries = 3
    retry_count = 0
    base_delay = 5

    while retry_count <= max_retries:
        try:
            completion = rate_limited_api_call(
                client.beta.chat.completions.parse,
                model="large",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.1,
                response_format=CorporateActionsData,
            )

            log_api_call(
                system_prompt,
                user_prompt,
                "large",
                completion,
                response_type="corporate_actions",
                page_num=page_num,
                company_name=company_name,
            )

            result = get_parsed_content(completion, CorporateActionsData)

            result.page_num = page_num

            if result.found_data:
                logger.info(
                    f"Corporate actions data found on page {page_num} for {company_name}"
                )

                actions_found = []
                if result.mergers_acquisitions:
                    actions_found.append("mergers_acquisitions")
                    if result.ma_details:
                        logger.info(f"M&A details: {result.ma_details}")
                if result.share_buyback_plan:
                    actions_found.append("share_buyback_plan")
                if result.dividend_policy_changes:
                    actions_found.append("dividend_policy_changes")
                if result.capital_structure_changes:
                    actions_found.append("capital_structure_changes")
                if result.restructuring_plans:
                    actions_found.append("restructuring_plans")
                if result.new_product_launches:
                    actions_found.append("new_product_launches")
                    if result.new_product_names:
                        logger.info(f"New products: {result.new_product_names}")
                    if result.last_product_launched:
                        logger.info(f"Last product: {result.last_product_launched}")
                if result.esg_initiatives:
                    actions_found.append("esg_initiatives")
                    if result.esg_initiative_details:
                        logger.info(f"ESG details: {result.esg_initiative_details}")
                if result.ongoing_litigation:
                    actions_found.append("ongoing_litigation")
                    if result.litigation_details:
                        logger.info(f"Litigation details: {result.litigation_details}")

                logger.info(f"Corporate actions found: {', '.join(actions_found)}")
            else:
                logger.info(
                    f"No corporate actions data found on page {page_num} for {company_name}"
                )

            return result

        except Exception as e:
            error_message = str(e)
            retry_count += 1

            if "Connection error" in error_message and retry_count <= max_retries:
                delay = base_delay * (2 ** (retry_count - 1))

                logger.warning(
                    f"Connection error in corporate actions extraction. Retrying attempt {retry_count}/{max_retries} after {delay} seconds. Error: {error_message}"
                )
                time.sleep(delay)
            else:
                logger.error(
                    f"Error extracting corporate actions data from page {page_num}: {error_message}"
                )

                return CorporateActionsData(
                    page_num=page_num,
                    found_data=False,
                    reasoning=f"Failed to extract corporate actions data due to: {error_message}",
                    reasoning_step2="No strategic analysis possible due to extraction failure.",
                )


def extract_leadership_management_from_page(
    page_text: str, page_num: int, company_name: str
) -> LeadershipManagementData:
    system_prompt = """
    You are a corporate governance expert specializing in analyzing company leadership information.
    Your task is to carefully analyze the text of an annual report page and extract all available
    information about leadership changes, executive appointments, and management structure.
    
    # Leadership Information to Extract
    
    Focus on extracting these key leadership elements:
    
    1. Leadership Positions Changed
       - Definition: Any positions in company leadership that experienced personnel changes
       - Look for: Changes in C-suite positions, board roles, senior management
       - Example: "Jane Smith replaced John Davis as Chief Financial Officer"
       - Record: List of specific positions that changed (e.g., "CEO", "CFO", "Board Chair")
    
    2. Executive Appointments
       - Definition: New executives joining the company or current executives appointed to new roles
       - Look for: New hires, promotions, internal moves to executive positions
       - Example: "The company appointed Dr. Michael Chen as the new Chief Technology Officer"
       - Record: List with details of appointments (name, position, date if available)
    
    3. Executive Departures
       - Definition: Executives leaving their positions or the company
       - Look for: Resignations, retirements, terminations, transitions out of leadership
       - Example: "After 12 years with the company, COO Sarah Johnson announced her retirement"
       - Record: List with details of departures (name, position, reason if stated)
    
    4. Board Changes
       - Definition: Changes to the board of directors composition or structure
       - Look for: New board members, departing directors, changes in board committees
       - Example: "Three new independent directors joined the Board this year"
       - Record: List of changes to board composition or structure
    
    5. Organizational Restructuring
       - Definition: Significant changes to reporting structures or leadership organization
       - Look for: Reorganization of divisions, creation of new leadership positions
       - Example: "The company consolidated its regional leadership into a global structure"
       - Record: Boolean (true/false) indicating if restructuring was mentioned
    
    # Extraction Guidelines
    
    1. DETAILED IDENTIFICATION:
       - For each category, identify specific changes mentioned on the page
       - Include names, positions, dates, and relevant details when available
       - Only extract leadership changes occurring during the current reporting period
    
    2. POSITION SPECIFICITY:
       - Be precise about the exact leadership positions mentioned
       - Distinguish between executive management and board positions
       - Note specific titles and roles (e.g., "Chief Marketing Officer" not just "executive")
    
    3. CHANGE CATEGORIZATION:
       - Correctly categorize each change as an appointment, departure, or position change
       - For complex situations (e.g., one executive replacing another), record information in both categories
       - Note whether changes were planned transitions or unexpected departures
    
    4. EVIDENCE CITATION:
       - Support each extraction with direct quotes from the text
       - Note the specific context surrounding each leadership change
       - Explain your confidence level for each extraction
    
    5. LEADERSHIP ANALYSIS:
       - For significant changes, provide brief analysis of potential implications
       - Consider how leadership changes might reflect or impact company strategy
       - Note any patterns or trends in leadership turnover
    
    6. DATA GAPS:
       - If no leadership information is found, leave the fields empty
       - Do not guess or infer changes that aren't clearly stated
       - Only extract information that appears directly on this specific page
    
    # Output Requirements
    
    1. Set found_data=True ONLY if you find evidence of at least one leadership change on the page
    2. For each category, provide a list of specific changes with relevant details
    3. Include detailed reasoning explaining how you identified each leadership change
    4. Add a second level of reasoning that analyzes the implications of these changes
    5. For categories where no changes are found, leave the fields as null/None
    """

    user_prompt = f"""
    Analyze the following page {page_num} from {company_name}'s annual report and extract all information about leadership changes and management.
    
    PAGE TEXT:
    {page_text}
    
    For each leadership change found on this page:
    1. Identify the specific position and nature of the change
    2. Extract details including names, positions, dates, and reasons when available
    3. Provide direct quotes from the text supporting your extraction
    4. Add analysis of the potential implications of these leadership changes
    
    Set found_data=True only if you find evidence of at least one leadership change on this page.
    """

    logger.info(f"Extracting leadership data from page {page_num} for {company_name}")

    max_retries = 3
    retry_count = 0
    base_delay = 5

    while retry_count <= max_retries:
        try:
            completion = rate_limited_api_call(
                client.beta.chat.completions.parse,
                model="large",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.1,
                response_format=LeadershipManagementData,
            )

            log_api_call(
                system_prompt,
                user_prompt,
                "large",
                completion,
                response_type="leadership_management",
                page_num=page_num,
                company_name=company_name,
            )

            result = get_parsed_content(completion, LeadershipManagementData)

            result.page_num = page_num

            if result.found_data:
                logger.info(
                    f"Leadership data found on page {page_num} for {company_name}"
                )

                if result.leadership_positions_changed:
                    logger.info(
                        f"Leadership positions changed: {result.leadership_positions_changed}"
                    )
                if result.executive_appointments:
                    logger.info(
                        f"Executive appointments: {result.executive_appointments}"
                    )
                if result.executive_departures:
                    logger.info(f"Executive departures: {result.executive_departures}")
                if hasattr(result, "board_changes") and result.board_changes:
                    logger.info(f"Board changes: {result.board_changes}")
                if (
                    hasattr(result, "organizational_restructuring")
                    and result.organizational_restructuring
                ):
                    logger.info(f"Organizational restructuring mentioned: Yes")
            else:
                logger.info(
                    f"No leadership data found on page {page_num} for {company_name}"
                )

            return result

        except Exception as e:
            error_message = str(e)
            retry_count += 1

            if "Connection error" in error_message and retry_count <= max_retries:
                delay = base_delay * (2 ** (retry_count - 1))

                logger.warning(
                    f"Connection error in leadership data extraction. Retrying attempt {retry_count}/{max_retries} after {delay} seconds. Error: {error_message}"
                )
                time.sleep(delay)
            else:
                logger.error(
                    f"Error extracting leadership data from page {page_num}: {error_message}"
                )

                return LeadershipManagementData(
                    page_num=page_num,
                    found_data=False,
                    reasoning=f"Failed to extract leadership information due to: {error_message}",
                    reasoning_step2="No leadership analysis possible due to extraction failure.",
                )


def search_relevant_pages(
    queries: List[str],
    company: Optional[str] = None,
    domain: Optional[str] = None,
    result_size: int = 8,
) -> List[Dict]:
    logger.info(f"Searching OpenSearch with {len(queries)} diverse query expansions")
    logger.info(f"Query details: {queries}")
    logger.info(f"Filters: company='{company}', domain='{domain}'")

    all_results = []
    seen_pages = set()

    try:
        for i, query in enumerate(queries):
            logger.info(f"Processing query {i + 1}: '{query}'")

            bool_query = {
                "must": [
                    {"match": {"text": {"query": query, "analyzer": "rebuilt_english"}}}
                ]
            }

            if company:
                if "filter" not in bool_query:
                    bool_query["filter"] = []
                bool_query["filter"].append({"term": {"company": company}})

            if domain:
                domain_boosts = {
                    "financial": [
                        "revenue",
                        "profit",
                        "income",
                        "earnings",
                        "margin",
                        "assets",
                        "liabilities",
                        "equity",
                        "cash flow",
                        "balance sheet",
                        "income statement",
                        "statement of operations",
                        "financial position",
                        "eps",
                        "ebitda",
                        "net income",
                        "gross profit",
                        "operating margin",
                        "dividend",
                        "fiscal year",
                        "quarter",
                        "annual",
                        "consolidated",
                        "financial highlights",
                        "financial review",
                        "financial results",
                        "management discussion",
                        "md&a",
                        "financial statements",
                        "notes to consolidated",
                        "independent auditor",
                        "accounting policies",
                    ],
                    "operations": [
                        "employees",
                        "workforce",
                        "headcount",
                        "staff",
                        "personnel",
                        "facilities",
                        "locations",
                        "operations",
                        "capacity",
                        "infrastructure",
                        "production",
                        "output",
                        "efficiency",
                        "utilization",
                        "throughput",
                        "stores",
                        "branches",
                        "fulfillment",
                        "distribution",
                        "logistics",
                        "supply chain",
                        "inventory",
                        "customers",
                        "users",
                        "subscribers",
                        "operational review",
                        "operations overview",
                        "business segments",
                        "key performance indicators",
                        "kpis",
                        "operational highlights",
                        "segment information",
                        "business overview",
                    ],
                    "corporate": [
                        "merger",
                        "acquisition",
                        "buyback",
                        "repurchase",
                        "dividend policy",
                        "restructuring",
                        "reorganization",
                        "strategic",
                        "initiative",
                        "transformation",
                        "strategy",
                        "vision",
                        "mission",
                        "outlook",
                        "future",
                        "growth",
                        "expansion",
                        "diversification",
                        "innovation",
                        "development",
                        "ceo letter",
                        "chairman statement",
                        "shareholder letter",
                        "corporate developments",
                        "corporate announcements",
                        "strategic review",
                    ],
                    "leadership": [
                        "leadership",
                        "executive",
                        "management",
                        "officer",
                        "director",
                        "board",
                        "committee",
                        "ceo",
                        "cfo",
                        "coo",
                        "chairman",
                        "appointed",
                        "appointment",
                        "joined",
                        "resigned",
                        "departed",
                        "retirement",
                        "succession",
                        "transition",
                        "promoted",
                        "stepping down",
                        "corporate governance",
                        "leadership team",
                        "executive management",
                        "board of directors",
                        "management committee",
                        "executive changes",
                        "leadership structure",
                        "management profiles",
                    ],
                }

                boost_terms = domain_boosts.get(domain, [])

                if boost_terms:
                    if "should" not in bool_query:
                        bool_query["should"] = []

                    for term in boost_terms:
                        bool_query["should"].append(
                            {"match": {"text": {"query": term, "boost": 2.0}}}
                        )

                    bool_query["minimum_should_match"] = 0

            search_body = {
                "query": {"bool": bool_query},
                "size": result_size,
                "_source": ["text", "company", "page_number"],
            }

            response = opensearch_client.search(index=INDEX_NAME, body=search_body)

            hits = response["hits"]["hits"]
            total_hits = (
                response["hits"]["total"]["value"]
                if "total" in response["hits"]
                else len(hits)
            )
            query_results_count = 0

            logger.info(f"Query {i + 1} found {total_hits} total hits in OpenSearch")

            for hit in hits:
                source = hit["_source"]
                page_key = f"{source.get('company')}_{source.get('page_number')}"

                if page_key not in seen_pages:
                    seen_pages.add(page_key)
                    query_results_count += 1

                    result = {
                        "score": hit["_score"],
                        "company": source.get("company"),
                        "page_number": source.get("page_number"),
                        "text": source.get("text"),
                        "query_index": i,  
                        "rank_in_query": len(
                            [r for r in all_results if r["query_index"] == i]
                        )
                        + 1,
                    }
                    all_results.append(result)

            logger.info(
                f"Query {i + 1} added {query_results_count} unique pages to results"
            )

        all_results.sort(key=lambda x: x["score"], reverse=True)

        logger.info(
            f"Total unique relevant pages found across all queries: {len(all_results)}"
        )

        if all_results:
            top_samples = min(3, len(all_results))
            for i in range(top_samples):
                result = all_results[i]
                excerpt = (
                    result["text"][:100] + "..."
                    if len(result["text"]) > 100
                    else result["text"]
                )
                logger.info(
                    f"Top result {i + 1}: Company={result['company']}, Page={result['page_number']}, Score={result['score']:.2f}, Excerpt: {excerpt}"
                )

        return all_results

    except Exception as e:
        logger.error(f"Error searching OpenSearch: {str(e)}")
        return []


def extract_metric_value(extracted_data: Dict, metric_type: str) -> Optional[float]:
    logger.info(f"Extracting {metric_type} from company data")

    if "pages_data" not in extracted_data or not extracted_data["pages_data"]:
        logger.info(f"No pages data available for metric extraction")
        return None

    for page_data in extracted_data.get("pages_data", []):
        if hasattr(page_data, metric_type):
            value = getattr(page_data, metric_type)
            if value is not None:
                logger.info(
                    f"Found {metric_type} value: {value} on page {getattr(page_data, 'page_num', 'unknown')}"
                )
                return value

        elif (
            isinstance(page_data, dict)
            and metric_type in page_data
            and page_data[metric_type] is not None
        ):
            value = page_data[metric_type]
            logger.info(
                f"Found {metric_type} value: {value} on page {page_data.get('page_num', 'unknown')}"
            )
            return value

    logger.info(f"No value found for {metric_type} in extracted data")
    return None


def process_companies_in_batch(
    search_queries: List[str],
    company_names: List[str],
    domain: str,
    metric_type: str,
    max_workers=3,
):
    logger.info(
        f"Batch processing {len(company_names)} companies for {metric_type} comparison"
    )
    logger.info(f"Companies to compare: {company_names}")
    logger.info(f"Using domain: {domain}, metric: {metric_type}")

    company_results = {}

    with tqdm(
        total=len(company_names), desc="Processing companies", unit="company"
    ) as company_pbar:
        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_company = {}
            for company_name in company_names:
                future = executor.submit(
                    process_document_by_domain, search_queries, company_name, domain
                )
                future_to_company[future] = company_name

            for future in concurrent.futures.as_completed(future_to_company):
                company_name = future_to_company[future]
                try:
                    extracted_data = future.result()
                    metric_value = extract_metric_value(extracted_data, metric_type)
                    company_results[company_name] = metric_value

                    if metric_value is not None:
                        logger.info(
                            f"Processed company {company_name}: {metric_type} = {metric_value}"
                        )
                    else:
                        logger.info(
                            f"Processed company {company_name}: {metric_type} not found"
                        )

                except Exception as e:
                    logger.error(f"Error processing company {company_name}: {str(e)}")
                    company_results[company_name] = None

                company_pbar.update(1)

    companies_with_data = sum(
        1 for value in company_results.values() if value is not None
    )
    logger.info(
        f"Comparison batch processing complete: {companies_with_data}/{len(company_names)} companies have {metric_type} data"
    )

    return company_results


def formulate_comparison_answer(comparison_data: Dict, question_kind: str) -> Answer:
    system_prompt = """
    You are an expert financial and business analyst specializing in comparing metrics across different companies from annual reports.
    Your task is to analyze the provided comparison data and formulate a precise answer through a structured reasoning process.
    
    # REASONING FRAMEWORK
    
    Your analysis must follow these specific reasoning steps:
    
    ## 1. QUESTION ANALYSIS
    - Dissect the comparison question into its core components
    - Identify precisely what metric is being compared between which companies
    - Determine the comparison type (highest/lowest/ranking/difference)
    - Clarify any ambiguities in the question
    - Identify special conditions (e.g., currency specifications, time periods)
    
    ## 2. DATA EXTRACTION & VALIDATION
    - Systematically examine the data available for each company
    - Verify which companies have valid data points and which have missing values
    - Check that all values are in comparable units and represent the same time periods
    - Note any data normalization that would be required for fair comparison
    - Organize the data in a structured format that facilitates comparison
    
    ## 3. COMPARISON METHODOLOGY
    - Establish a clear framework for company comparison
    - Rank companies based on the metric values where appropriate
    - Identify significant differences between companies
    - Calculate relevant statistics (e.g., percent differences, averages)
    - Apply appropriate sorting or filtering based on the question
    
    ## 4. MATHEMATICAL CALCULATIONS (when needed)
    - Show explicit step-by-step calculations for any derived values
    - Explain the mathematical rationale behind each calculation
    - Verify results through cross-checking methods
    - Present calculations with appropriate precision and units
    
    ## 5. SOURCE ASSESSMENT
    - Evaluate the reliability and completeness of the available data
    - Identify potential limitations or biases in the source data
    - Assess whether there are sufficient data points to answer with confidence
    - Note what additional information would strengthen the analysis
    
    ## 6. CONFIDENCE EVALUATION
    - Assess confidence level (0-1) based on:
      * Data quality and completeness
      * Consistency across sources
      * Clarity of the question
      * Precision of available metrics
    - Justify your confidence assessment
    
    ## 7. FINAL DETERMINATION
    - State your final answer clearly and directly
    - Explain why this answer best addresses the comparison question
    - Address any remaining uncertainties or caveats
    - Format the answer according to the expected question type
    
    # ANSWER FORMATS
    
    Format your final answer value based on the question type:
    
    - For 'name' questions: Return the name of the company that best answers the question
      * Example: "Acme Corporation"
    
    - For 'names' questions: Return an ordered list of company names
      * Example: ["Acme Corporation", "TechGiant Inc.", "Global Services Ltd."]
    
    - For 'number' questions: Return the precise numerical value with appropriate units
      * Example: 42.7 (for a numeric value), "N/A" (if data insufficient)
    
    - For 'boolean' questions: Return true or false
      * Example: true (if comparison condition is met), false (if not met or insufficient data)
    
    # DATA SUFFICIENCY RULES
    
    When determining if the available data is sufficient:
    
    1. For comparative questions (highest/lowest/ranking):
      - Need valid data for at least two companies to make a comparison
      - If only one company has data, return "N/A" for number questions or the appropriate default for other types
    
    2. For specific difference calculations:
      - Need valid data for all companies mentioned in the specific comparison
      - If any company in the direct comparison lacks data, return "N/A" or appropriate default
    
    3. For threshold questions (e.g., "Does any company exceed X?"):
      - Need at least one valid data point to potentially answer "true"
      - Default to "false" if no companies have valid data
    """

    user_prompt = f"""
    Analyze the following comparison data and formulate a comprehensive answer with structured reasoning.
    
    QUESTION: {comparison_data["question"]}
    QUESTION TYPE: {question_kind}
    METRIC TYPE: {comparison_data["metric_type"]}
    
    COMPANY RESULTS:
    {json.dumps(comparison_data["company_results"], indent=2)}
    
    For each step in your reasoning:
    1. Provide detailed analysis specific to this comparison scenario
    2. Address the particular companies and metric being compared
    3. Consider any special conditions mentioned in the question
    4. Format your final answer according to the question type
    
    If the data is insufficient to answer the question with confidence, clearly explain why
    and return "N/A" or the appropriate default value based on the question type.
    """

    logger.info(
        f"Formulating comparison answer for question: {comparison_data['question']}"
    )
    logger.info(
        f"Question type: {question_kind}, Metric: {comparison_data['metric_type']}"
    )

    valid_companies = sum(
        1 for value in comparison_data["company_results"].values() if value is not None
    )
    logger.info(
        f"Valid data points available for {valid_companies} out of {len(comparison_data['company_results'])} companies"
    )

    max_retries = 3
    retry_count = 0
    base_delay = 5

    while retry_count <= max_retries:
        try:
            completion = rate_limited_api_call(
                client.beta.chat.completions.parse,
                model="large",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.1,
                response_format=Answer,
            )

            log_api_call(
                system_prompt,
                user_prompt,
                "large",
                completion,
                response_type="comparison_answer",
            )

            result = get_parsed_content(completion, Answer)

            logger.info(
                f"Comparison answer formulated with confidence level: {result.confidence}"
            )
            logger.info(f"Answer value: {result.value}")

            return result

        except Exception as e:
            error_message = str(e)
            retry_count += 1

            if "Connection error" in error_message and retry_count <= max_retries:
                delay = base_delay * (2 ** (retry_count - 1))

                logger.warning(
                    f"Connection error in comparison answer formulation. Retrying attempt {retry_count}/{max_retries} after {delay} seconds. Error: {error_message}"
                )
                time.sleep(delay)
            else:
                logger.error(f"Error formulating comparison answer: {error_message}")

                default_value = "N/A"
                if question_kind == "boolean":
                    default_value = False
                elif question_kind == "names":
                    default_value = ["N/A"]

                return Answer(
                    reasoning_question_analysis=f"The question asks for a comparison of {comparison_data['metric_type']} across multiple companies.",
                    reasoning_context_evaluation=f"Error occurred during processing: {error_message}. Unable to properly evaluate comparison data.",
                    reasoning_final_decision=f"Due to processing error, cannot complete comparison analysis. Defaulting to {default_value} as the answer.",
                    value=default_value,
                    confidence=0.0,
                    sources=None,
                )


def process_comparison_question(
    question_text: str,
    question_kind: str,
    company_names: List[str],
    search_queries: List[str],
) -> Answer:
    logger.info(f"Processing comparison question: {question_text}")
    logger.info(f"Company names to compare: {company_names}")

    metric_type = determine_metric_type(question_text)
    logger.info(f"Determined metric type for comparison: {metric_type}")

    domain = determine_domain_for_metric(metric_type)
    logger.info(f"Determined domain for comparison: {domain}")

    logger.info(
        f"Processing {len(company_names)} companies in batch for {metric_type} comparison"
    )
    company_results = process_companies_in_batch(
        search_queries, company_names, domain, metric_type
    )

    comparison_data = {
        "question": question_text,
        "metric_type": metric_type,
        "company_results": company_results,
    }

    valid_results = {
        company: value
        for company, value in company_results.items()
        if value is not None
    }
    logger.info(
        f"Comparison data collected for {len(valid_results)}/{len(company_names)} companies"
    )
    if valid_results:
        logger.info(f"Sample values: {list(valid_results.items())[:2]}")

    return formulate_comparison_answer(comparison_data, question_kind)


def answer_question(
    question_text: str,
    question_kind: str,
    domain: str,
    extracted_data: Dict,
    company_name: str,
) -> Answer:
    system_prompt = """
    You are an expert financial analyst specializing in extracting insights from annual reports.
    Your task is to analyze the provided information from a company's annual report and formulate
    a precise answer through a structured reasoning process.
    
    # COMPREHENSIVE REASONING FRAMEWORK
    
    Your analysis must follow these specific reasoning steps:
    
    ## 1. QUESTION ANALYSIS
    - Dissect the question into its core components
    - Identify the specific information being requested about the company
    - Determine what time period is relevant (current year, previous year, specific date)
    - Clarify how to interpret phrases like "at the end of the period" or "within the last period"
    - Note any currency specifications or unit requirements
    - Specify what form the answer should take (numerical, textual, boolean, etc.)
    
    ## 2. CONTEXT EVALUATION
    - Methodically examine each page/section of the provided extracted data
    - Identify relevant passages, tables, figures, or statements across all sources
    - Compare information across different pages, noting agreements and discrepancies
    - Assess the reliability and authority of different information sources
    - Evaluate the completeness of the available information
    - Consider how information gaps affect your ability to answer with confidence
    
    ## 3. SOURCE COMPARISON
    - When multiple pages contain relevant information, compare them systematically
    - Identify which sources are most authoritative or recent
    - Resolve any contradictions between different pages
    - Consider how different sections of the annual report might present the same information differently
    - Determine which source(s) should take precedence for your answer
    
    ## 4. TEMPORAL ANALYSIS
    - For time-sensitive questions, identify which reporting period(s) the question refers to
    - Clarify how terms like "end of period" apply to this company's fiscal calendar
    - Analyze year-over-year or quarter-over-quarter changes if relevant
    - Consider seasonality or cyclical factors that may affect interpretation
    - Ensure you're using data from the correct time period to answer the question
    
    ## 5. DATA NORMALIZATION
    - When comparing metrics across time periods or categories, ensure proper normalization
    - Verify units are consistent (e.g., millions vs. billions, USD vs. EUR)
    - Note when currency conversion is necessary and how it was performed
    - Adjust for any accounting changes or restatements mentioned in the report
    - Ensure fair comparison when context differs across data points
    
    ## 6. MATHEMATICAL CALCULATION
    - When calculations are needed, show each step explicitly
    - Explain the reasoning behind each calculation
    - Verify calculations through cross-checking methods
    - Ensure units and time periods are consistent throughout calculations
    - Interpret the significance of the calculation results in business context
    
    ## 7. CONFIDENCE ASSESSMENT
    - Evaluate your confidence level (0-1) based on:
      * Quality and completeness of available data
      * Consistency across sources
      * Clarity of the question
      * Precision of available metrics
    - Explain factors that increase or decrease confidence
    - Identify what additional information would improve confidence
    
    ## 8. FINAL DETERMINATION
    - Synthesize all previous analysis into a conclusive answer
    - Consider multiple possible interpretations and explain why your chosen answer is best
    - Address any remaining uncertainties or caveats
    - Format the answer according to the expected question type
    - Cite specific sources (page numbers and quotes) that support your conclusion
    
    # SOURCE CITATION
    
    When citing information from the extracted data:
    - Specify the page number where the information appears
    - Include a brief relevant quote that supports your point
    - Format as: [Page X: "relevant excerpt"]
    - Prioritize direct quotes from financial statements or management discussion
    
    # ANSWER FORMATS
    
    Format your final answer value based on the question type:
    
    - For 'name' questions: Return the specific name (e.g., product, executive)
      * Example: "GreenTech 3000"
    
    - For 'names' questions: Return a list of names
      * Example: ["GreenTech 3000", "EcoSolution X", "NatureFriendly Pro"]
    
    - For 'number' questions: Return the precise numerical value with appropriate units
      * Example: 42.7 (for a numeric value), "N/A" (if data insufficient)
    
    - For 'boolean' questions: Return true or false
      * Example: true (if condition is met), false (if not met or insufficient data)
    
    # DATA SUFFICIENCY RULES
    
    When determining if the available data is sufficient:
    
    1. For specific value questions:
      - Need a clear, direct statement or figure from the report
      - If the exact information is not found but can be reliably calculated, show your calculation
      - If the required information is not available, return "N/A" for number questions
    
    2. For yes/no questions:
      - Need explicit evidence to answer "true"
      - Default to "false" if no relevant information is found
      - Clearly explain when defaulting due to lack of information
    
    3. For list questions:
      - If no relevant items are found, return ["N/A"]
      - Only include items explicitly mentioned in the extracted data
    
    # DOMAIN-SPECIFIC CONSIDERATIONS
    
    Adapt your analysis based on the question domain:
    
    - Financial domain: Pay special attention to accounting terminology, fiscal periods, and financial statement context
    - Operations domain: Focus on quantitative business metrics, operational capacity, and market positioning
    - Corporate domain: Analyze strategic decisions, corporate actions, and their business implications
    - Leadership domain: Examine management structure, executive changes, and governance implications
    - Other domains: Consider broader business context and industry-specific factors
    """

    extracted_json = json.dumps(
        extracted_data,
        default=lambda o: o.model_dump()
        if hasattr(o, "model_dump")
        else (o.dict() if hasattr(o, "dict") else str(o)),
        indent=2,
    )

    user_prompt = f"""
    Answer the following question about {company_name} based on the extracted data from its annual report.
    
    QUESTION: {question_text}
    QUESTION TYPE: {question_kind}
    QUESTION DOMAIN: {domain}
    
    EXTRACTED DATA:
    {extracted_json}
    
    Provide a comprehensive answer following all the structured reasoning steps:
    1. Analyze what the question is asking and what answer is required
    2. Evaluate the available information across all extracted pages
    3. Compare information from different sources when available
    4. Analyze temporal aspects for time-sensitive questions
    5. Normalize data for consistency when needed
    6. Perform calculations with explicit steps when required
    7. Assess your confidence level based on the available evidence
    8. Provide your final determination with source citations
    
    If the extracted data does not contain sufficient information to answer the question,
    clearly explain what's missing and return "N/A" or the appropriate default value.
    
    Remember to cite specific page numbers and quotes that support your answer.
    """

    logger.info(f"Formulating answer to question about {company_name}: {question_text}")
    logger.info(f"Question type: {question_kind}, Domain: {domain}")

    page_count = len(extracted_data.get("pages_data", []))
    logger.info(f"Answer will be based on {page_count} pages with relevant data")

    max_retries = 3
    retry_count = 0
    base_delay = 5

    while retry_count <= max_retries:
        try:
            completion = rate_limited_api_call(
                client.beta.chat.completions.parse,
                model="large",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.1,
                response_format=Answer,
            )

            log_api_call(
                system_prompt,
                user_prompt,
                "large",
                completion,
                response_type="answer",
                company_name=company_name,
            )

            result = get_parsed_content(completion, Answer)

            logger.info(f"Answer formulated with confidence level: {result.confidence}")
            logger.info(f"Answer value: {result.value}")

            if result.sources and len(result.sources) > 0:
                source_pages = [source.page_number for source in result.sources]
                logger.info(
                    f"Answer based on {len(result.sources)} source(s), pages: {source_pages}"
                )

            return result

        except Exception as e:
            error_message = str(e)
            retry_count += 1

            if "Connection error" in error_message and retry_count <= max_retries:
                delay = base_delay * (2 ** (retry_count - 1))

                logger.warning(
                    f"Connection error in answer formulation. Retrying attempt {retry_count}/{max_retries} after {delay} seconds. Error: {error_message}"
                )
                time.sleep(delay)
            else:
                logger.error(f"Error formulating answer: {error_message}")

                default_value = "N/A"
                if question_kind == "boolean":
                    default_value = False
                elif question_kind == "names":
                    default_value = ["N/A"]

                return Answer(
                    reasoning_question_analysis=f"The question asks for information about {domain} aspects of {company_name}.",
                    reasoning_context_evaluation=f"Error occurred during processing: {error_message}. Unable to properly evaluate the extracted data.",
                    reasoning_final_decision=f"Due to processing error, cannot complete analysis. Defaulting to {default_value} as the answer.",
                    value=default_value,
                    confidence=0.0,
                    sources=None,
                )


def process_document_by_domain(
    search_queries: List[str],
    company_name: str,
    domain: str,
    result_size_per_query: int = 8,
) -> Dict:
    logger.info(f"Processing document extraction for {company_name}, domain: {domain}")
    logger.info(
        f"Using {len(search_queries)} search query expansions: {search_queries}"
    )

    search_results = search_relevant_pages(
        queries=search_queries,
        company=company_name,
        domain=domain,
        result_size=result_size_per_query,
    )

    extracted_data = {
        "domain": domain,
        "company_name": company_name,
        "pages_data": [],
        "search_queries_used": search_queries,
        "timestamp": datetime.datetime.now().isoformat(),
    }

    if not search_results:
        logger.warning(f"No relevant pages found for {company_name}, domain {domain}")
        return extracted_data

    logger.info(f"Preparing {len(search_results)} pages for information extraction")
    pages_data = []
    for page_result in search_results:
        pages_data.append(
            {
                "text": page_result["text"],
                "page_number": page_result["page_number"],
                "company_name": company_name,
                "query_index": page_result.get("query_index", -1),
                "score": page_result.get("score", 0),
            }
        )

    batch_size = 5
    max_workers = 3

    logger.info(
        f"Processing pages in batches: batch_size={batch_size}, max_workers={max_workers}"
    )

    if domain == "financial":
        logger.info(f"Extracting financial metrics from {len(pages_data)} pages")
        extracted_data["pages_data"] = process_pages_in_batch(
            extract_financial_data_from_page, pages_data, batch_size, max_workers
        )
        logger.info(
            f"Extracted financial data from {len(extracted_data['pages_data'])} pages"
        )

    elif domain == "operations":
        logger.info(
            f"Extracting business operations metrics from {len(pages_data)} pages"
        )
        extracted_data["pages_data"] = process_pages_in_batch(
            extract_business_operations_from_page, pages_data, batch_size, max_workers
        )
        logger.info(
            f"Extracted operations data from {len(extracted_data['pages_data'])} pages"
        )

    elif domain == "corporate":
        logger.info(f"Extracting corporate actions from {len(pages_data)} pages")
        extracted_data["pages_data"] = process_pages_in_batch(
            extract_corporate_actions_from_page, pages_data, batch_size, max_workers
        )
        logger.info(
            f"Extracted corporate actions data from {len(extracted_data['pages_data'])} pages"
        )

    elif domain == "leadership":
        logger.info(f"Extracting leadership information from {len(pages_data)} pages")
        extracted_data["pages_data"] = process_pages_in_batch(
            extract_leadership_management_from_page, pages_data, batch_size, max_workers
        )
        logger.info(
            f"Extracted leadership data from {len(extracted_data['pages_data'])} pages"
        )

    else:
        logger.info(
            f"Domain '{domain}' not specifically recognized, trying all extractors"
        )

        financial_data = process_pages_in_batch(
            extract_financial_data_from_page, pages_data, batch_size, max_workers
        )
        logger.info(f"Extracted financial data from {len(financial_data)} pages")

        operations_data = process_pages_in_batch(
            extract_business_operations_from_page, pages_data, batch_size, max_workers
        )
        logger.info(f"Extracted operations data from {len(operations_data)} pages")

        corporate_data = process_pages_in_batch(
            extract_corporate_actions_from_page, pages_data, batch_size, max_workers
        )
        logger.info(
            f"Extracted corporate actions data from {len(corporate_data)} pages"
        )

        leadership_data = process_pages_in_batch(
            extract_leadership_management_from_page, pages_data, batch_size, max_workers
        )
        logger.info(f"Extracted leadership data from {len(leadership_data)} pages")

        extracted_data["pages_data"] = (
            financial_data + operations_data + corporate_data + leadership_data
        )
        logger.info(
            f"Combined data from all domains: {len(extracted_data['pages_data'])} total pages with data"
        )

    extracted_data["pages_count"] = len(extracted_data["pages_data"])
    extracted_data["extraction_success"] = len(extracted_data["pages_data"]) > 0

    logger.info(
        f"Document processing complete: found relevant data on {len(extracted_data['pages_data'])} pages"
    )

    return extracted_data


def create_reference_from_source(
    source: Source, company_name: str
) -> Optional[Reference]:
    global company_to_sha1

    sha1 = company_to_sha1.get(company_name)
    if not sha1:
        logger.warning(f"SHA1 hash not found for company '{company_name}' in metadata")

        for known_company, known_sha1 in company_to_sha1.items():
            if (
                company_name.lower() in known_company.lower()
                or known_company.lower() in company_name.lower()
            ):
                logger.info(
                    f"Using SHA1 from similar company name: '{known_company}' for '{company_name}'"
                )
                sha1 = known_sha1
                break

        if not sha1:
            return None

    return Reference(
        pdf_sha1=sha1,
        page_index=source.page_number,
        excerpt_text=source.text_excerpt[:200]
        if hasattr(source, "text_excerpt") and source.text_excerpt
        else None,
    )


def format_output_value(value, question_kind: str):
    logger.info(
        f"Formatting output value of type {type(value)} for question_kind: {question_kind}"
    )

    if value == "N/A":
        if question_kind == "number":
            return "N/A"
        elif question_kind == "boolean":
            return "False"
        elif question_kind == "name":
            return "N/A"
        elif question_kind == "names":
            return ["N/A"]
        return value

    if question_kind == "boolean":
        if isinstance(value, bool):
            return str(value).capitalize()
        elif isinstance(value, str) and value.lower() in ["true", "false"]:
            return value.capitalize()
        else:
            logger.warning(f"Invalid boolean value: {value}, defaulting to False")
            return "False"

    elif question_kind == "number":
        if isinstance(value, (int, float)):
            return value
        elif isinstance(value, str) and value.replace(".", "", 1).isdigit():
            try:
                return float(value)
            except ValueError:
                logger.warning(
                    f"Failed to convert string '{value}' to number, returning as is"
                )
                return value
        else:
            logger.warning(
                f"Non-numeric value for number question: {value}, returning 'N/A'"
            )
            return "N/A"

    elif question_kind == "name":
        if isinstance(value, (list, tuple)) and len(value) > 0:
            logger.warning(f"List value for name question: {value}, taking first item")
            return str(value[0])
        else:
            return str(value)

    elif question_kind == "names":
        if isinstance(value, (list, tuple)):
            return [str(item) for item in value]
        else:
            logger.warning(
                f"Non-list value for names question: {value}, wrapping in list"
            )
            return [str(value)]

    logger.info(f"No specific formatting applied, returning value as is")
    return value


def process_annual_report_questions(
    questions_file: str, meta_file: str
) -> List[OutputAnswer]:
    logger.info(f"Starting processing of questions from file: {questions_file}")
    logger.info(f"Using metadata from file: {meta_file}")

    global company_to_sha1
    company_to_sha1 = load_company_metadata(meta_file)

    if not company_to_sha1:
        logger.error(
            "Failed to load company metadata. Cannot proceed without SHA1 mappings."
        )
        return []

    try:
        with open(questions_file, "r", encoding="utf-8") as f:
            questions_data = json.load(f)

        logger.info(
            f"Successfully loaded {len(questions_data)} questions for processing"
        )
    except Exception as e:
        logger.error(f"Error loading questions file: {str(e)}")
        return []

    company_names = get_company_names_from_opensearch()

    if not company_names:
        logger.error(
            "Failed to retrieve company names from OpenSearch. Cannot proceed."
        )
        return []

    answers = []
    extracted_info_cache = {}

    with tqdm(
        total=len(questions_data), desc="Processing questions", unit="question"
    ) as pbar:
        for q_idx, q_data in enumerate(questions_data):
            question_text = q_data["text"]
            question_kind = q_data["kind"]

            logger.info(f"\n\n--- Question {q_idx + 1}/{len(questions_data)} ---")
            logger.info(f"Text: {question_text}")
            logger.info(f"Type: {question_kind}")

            output_answer = OutputAnswer(
                question_text=question_text, value="N/A", confidence=0.0, references=[]
            )

            start_time = time.time()

            try:
                is_comparison = is_comparison_question(question_text)

                if is_comparison:
                    logger.info(f"Question classified as COMPARISON question")

                    comparison_companies = extract_company_names_from_question(
                        question_text, company_names
                    )

                    if not comparison_companies or len(comparison_companies) < 2:
                        logger.warning(
                            f"Insufficient companies found for comparison question. Found: {comparison_companies}"
                        )
                        answers.append(output_answer)
                        pbar.update(1)
                        continue

                    logger.info(f"Companies to compare: {comparison_companies}")

                    company_info = identify_company_and_domain(
                        question_text, question_kind, company_names
                    )
                    search_queries = [
                        company_info.query_expansion_1,
                        company_info.query_expansion_2,
                        company_info.query_expansion_3,
                    ]

                    answer_obj = process_comparison_question(
                        question_text,
                        question_kind,
                        comparison_companies,
                        search_queries,
                    )

                    output_answer.value = format_output_value(
                        answer_obj.value, question_kind
                    )
                    output_answer.confidence = answer_obj.confidence

                    if answer_obj.sources:
                        for source in answer_obj.sources:
                            source_company = source.document_name
                            if source_company.endswith(".pdf"):
                                source_company = source_company[:-4]

                            reference = create_reference_from_source(
                                source, source_company
                            )
                            if reference:
                                output_answer.references.append(reference)

                else:
                    logger.info(f"Question classified as SINGLE COMPANY question")

                    company_info = identify_company_and_domain(
                        question_text, question_kind, company_names
                    )

                    if not company_info.company:
                        logger.warning(
                            f"Could not identify company in question: {question_text}"
                        )
                        answers.append(output_answer)
                        pbar.update(1)
                        continue

                    company_name = company_info.company
                    domain = company_info.domain
                    logger.info(f"Identified company: {company_name}, domain: {domain}")

                    search_queries = [
                        company_info.query_expansion_1,
                        company_info.query_expansion_2,
                        company_info.query_expansion_3,
                    ]
                    logger.info(f"Query expansions generated:")
                    logger.info(f"1. Core: {company_info.query_expansion_1}")
                    logger.info(f"2. Alt: {company_info.query_expansion_2}")
                    logger.info(f"3. Context: {company_info.query_expansion_3}")

                    cache_key = f"{company_name}_{domain}_{'_'.join(search_queries)}"

                    if cache_key not in extracted_info_cache:
                        logger.info(
                            f"Extracting new data for {company_name}, domain {domain}"
                        )

                        extracted_data = process_document_by_domain(
                            search_queries, company_name, domain
                        )

                        extracted_info_cache[cache_key] = extracted_data
                    else:
                        logger.info(
                            f"Using cached data for {company_name}, domain {domain}"
                        )

                    extracted_data = extracted_info_cache[cache_key]

                    answer_obj = answer_question(
                        question_text,
                        question_kind,
                        domain,
                        extracted_data,
                        company_name,
                    )

                    output_answer.value = format_output_value(
                        answer_obj.value, question_kind
                    )
                    output_answer.confidence = answer_obj.confidence

                    if answer_obj.sources:
                        for source in answer_obj.sources:
                            reference = create_reference_from_source(
                                source, company_name
                            )
                            if reference:
                                output_answer.references.append(reference)

                answers.append(output_answer)

                elapsed_time = time.time() - start_time
                logger.info(f"Question processed in {elapsed_time:.2f} seconds")
                logger.info(f"Answer: {output_answer.value}")
                logger.info(f"Confidence: {output_answer.confidence}")
                logger.info(f"References: {len(output_answer.references)}")

            except Exception as e:
                logger.error(f"Error processing question {q_idx + 1}: {str(e)}")
                logger.exception("Exception details:")

                answers.append(output_answer)

            finally:
                pbar.update(1)

    logger.info(f"\n--- Processing Summary ---")
    logger.info(f"Total questions processed: {len(answers)}")

    questions_with_refs = sum(1 for answer in answers if answer.references)
    logger.info(
        f"Questions with references: {questions_with_refs} ({questions_with_refs / len(answers) * 100:.1f}%)"
    )

    high_confidence = sum(
        1
        for answer in answers
        if hasattr(answer, "confidence") and answer.confidence >= 0.7
    )
    medium_confidence = sum(
        1
        for answer in answers
        if hasattr(answer, "confidence") and 0.3 <= answer.confidence < 0.7
    )
    low_confidence = sum(
        1
        for answer in answers
        if hasattr(answer, "confidence") and answer.confidence < 0.3
    )

    logger.info(f"Confidence distribution:")
    logger.info(
        f"  High (≥0.7): {high_confidence} ({high_confidence / len(answers) * 100:.1f}%)"
    )
    logger.info(
        f"  Medium (0.3-0.7): {medium_confidence} ({medium_confidence / len(answers) * 100:.1f}%)"
    )
    logger.info(
        f"  Low (<0.3): {low_confidence} ({low_confidence / len(answers) * 100:.1f}%)"
    )

    return answers


def save_answers_to_file(answers: List[OutputAnswer], output_file: str):
    logger.info(f"Saving {len(answers)} answers to output file: {output_file}")

    try:
        serializable_answers = {"answers": pydantic_to_dict(answers)}

        output_dir = os.path.dirname(output_file)
        if output_dir and not os.path.exists(output_dir):
            os.makedirs(output_dir)
            logger.info(f"Created output directory: {output_dir}")

        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(serializable_answers, f, ensure_ascii=False, indent=2)

        logger.info(f"Successfully saved answers to: {output_file}")

        references_count = sum(len(answer.references) for answer in answers)
        answers_with_refs = sum(1 for answer in answers if answer.references)
        logger.info(
            f"Output includes {len(answers)} answers with {references_count} references"
        )
        logger.info(
            f"Answers with references: {answers_with_refs} ({answers_with_refs / len(answers) * 100:.1f}%)"
        )

        file_size_bytes = os.path.getsize(output_file)
        file_size_mb = file_size_bytes / (1024 * 1024)
        logger.info(f"Output file size: {file_size_mb:.2f} MB")

    except Exception as e:
        logger.error(f"Error saving answers to file: {str(e)}")
        logger.exception("Exception details:")


def main():
    questions_file = (
        "/Users/ruadhv7/Desktop/test/my_project/cluster/challange/data/questions.json"
    )
    meta_file = (
        "/Users/ruadhv7/Desktop/test/my_project/cluster/challange/data/meta.json"
    )
    output_file = "answers_enhanced_reasoning_.json"

    logger.info("=" * 80)
    logger.info("ANNUAL REPORT ANALYZER - ENHANCED VERSION")
    logger.info("=" * 80)
    logger.info(
        "Starting annual report processing with OpenSearch and advanced LLM classifiers"
    )

    if not os.path.exists(questions_file):
        logger.error(f"Error: Questions file {questions_file} does not exist")
        return

    if not os.path.exists(meta_file):
        logger.error(f"Error: Metadata file {meta_file} does not exist")
        return

    try:
        with jsonlines.open(api_log_file, mode="w") as writer:
            writer.write(
                {
                    "timestamp": datetime.datetime.now().isoformat(),
                    "type": "header",
                    "message": "Annual Report Analysis API Log - Sequential JSONL Format",
                }
            )
        logger.info(f"Initialized API log file: {api_log_file}")
    except Exception as e:
        logger.error(f"Error initializing API log file: {str(e)}")

    start_time = time.time()

    try:
        answers = process_annual_report_questions(questions_file, meta_file)

        save_answers_to_file(answers, output_file)

        elapsed_time = time.time() - start_time
        logger.info(
            f"Total processing time: {elapsed_time:.2f} seconds ({elapsed_time / 60:.2f} minutes)"
        )

        logger.info(f"Processed {len(answers)} questions successfully")

        logger.info("\nExample answers:")
        for i, answer in enumerate(answers[:3]):
            logger.info(f"{i + 1}. Question: {answer.question_text}")
            logger.info(f"   Answer: {answer.value}")
            logger.info(
                f"   Confidence: {answer.confidence if hasattr(answer, 'confidence') else 'N/A'}"
            )
            logger.info(f"   References: {len(answer.references)}")
            logger.info("")

        logger.info("Processing completed successfully!")
        logger.info("=" * 80)

    except Exception as e:
        logger.error(f"Error in main processing: {str(e)}")
        logger.exception("Exception details:")

        elapsed_time = time.time() - start_time
        logger.info(
            f"Execution terminated after {elapsed_time:.2f} seconds ({elapsed_time / 60:.2f} minutes)"
        )


if __name__ == "__main__":
    step_counter = 0

    main()
